
# OPTATIVO III — GESTIÓN DE LA CALIDAD DE LOS PROCESOS

## Proyecto: Red Salud Integral Carazo

### Objetivo
Evaluar y mejorar la calidad de los datos de citas médicas utilizados en el proceso de Minería de Datos, mediante análisis exploratorio, validaciones automáticas, indicadores de calidad, registro de incidencias y aplicación del ciclo PHVA.

### Fuente de información
- Base de datos: RedSaludCarazoDW_V2
- Tabla de origen: dbo.raw_cita
- Tabla analítica de referencia: dbo.dm_dataset_prediccion

### Indicadores establecidos en el II corte
1. Tasa de integridad de registros de citas.
2. Porcentaje de filas duplicadas.
3. Tasa de duplicidad de identificadores de cita.


In [1]:

# ============================================================
# OPTATIVO III - GESTIÓN DE LA CALIDAD DE LOS PROCESOS
# PASO 1. CONEXIÓN Y VERIFICACIÓN DE LA FUENTE DE DATOS
# ============================================================

import sys
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import text
from IPython.display import display


# ============================================================
# 1. LOCALIZAR EL PROYECTO
# ============================================================

rutas = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "RedSaludCarazo_proyecto_base"
]

proyecto = next(
    (
        ruta for ruta in rutas
        if (ruta / "src" / "database.py").is_file()
    ),
    None
)

if proyecto is None:
    raise FileNotFoundError(
        "No se encontró la carpeta del proyecto."
    )

print("Carpeta del proyecto:", proyecto)


# ============================================================
# 2. CONFIGURAR LA CONEXIÓN
# ============================================================

sys.path.insert(0, str(proyecto / "src"))

load_dotenv(
    proyecto / ".env",
    override=True
)

from database import engine_sqlserver

engine = engine_sqlserver()


# ============================================================
# 3. CONSULTAR SQL SERVER
# ============================================================

with engine.connect() as conexion:

    nombre_bd = conexion.execute(
        text("SELECT DB_NAME()")
    ).scalar_one()

    datos_originales = pd.read_sql_query(
        text("SELECT * FROM dbo.raw_cita"),
        conexion
    )

    datos_predictivos = pd.read_sql_query(
        text("SELECT * FROM dbo.dm_dataset_prediccion"),
        conexion
    )


# ============================================================
# 4. MOSTRAR INFORMACIÓN GENERAL
# ============================================================

print("\n" + "=" * 65)
print("RED SALUD INTEGRAL CARAZO")
print("VERIFICACIÓN DE LA FUENTE DE DATOS")
print("=" * 65)

print("\nBase de datos:", nombre_bd)

print("\nTABLA ORIGINAL")
print("Nombre: dbo.raw_cita")
print("Registros:", len(datos_originales))
print("Columnas:", len(datos_originales.columns))

print("\nTABLA ANALÍTICA")
print("Nombre: dbo.dm_dataset_prediccion")
print("Registros:", len(datos_predictivos))
print("Columnas:", len(datos_predictivos.columns))


# ============================================================
# 5. MOSTRAR ESTRUCTURA DE LA TABLA ORIGINAL
# ============================================================

print("\n" + "=" * 65)
print("ESTRUCTURA DE LOS DATOS ORIGINALES")
print("=" * 65)

estructura = pd.DataFrame({
    "Campo": datos_originales.columns,
    "Tipo de dato": datos_originales.dtypes.astype(str).values
})

display(estructura)


# ============================================================
# 6. MOSTRAR UNA MUESTRA
# ============================================================

print("\nPRIMEROS 5 REGISTROS ORIGINALES")

display(datos_originales.head(5))


# ============================================================
# 7. VERIFICAR IDENTIFICADORES
# ============================================================

ids = (
    datos_originales["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

frecuencias = ids.dropna().value_counts()

identificadores_distintos = int(
    ids.nunique()
)

grupos_repetidos = int(
    (frecuencias > 1).sum()
)

print("\n" + "=" * 65)
print("CONTROL INICIAL DE IDENTIFICADORES")
print("=" * 65)

print("Registros originales:", len(datos_originales))
print("Identificadores distintos:", identificadores_distintos)
print("Grupos de IDs repetidos:", grupos_repetidos)

print("\nPASO 1 FINALIZADO")
print("Consultas de solo lectura.")
print("No se modificaron datos de SQL Server.")


Carpeta del proyecto: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base

RED SALUD INTEGRAL CARAZO
VERIFICACIÓN DE LA FUENTE DE DATOS

Base de datos: RedSaludCarazoDW_V2

TABLA ORIGINAL
Nombre: dbo.raw_cita
Registros: 10080
Columnas: 19

TABLA ANALÍTICA
Nombre: dbo.dm_dataset_prediccion
Registros: 8996
Columnas: 13

ESTRUCTURA DE LOS DATOS ORIGINALES


,Campo,Tipo de dato
0,id_raw,int64
1,lote_carga,str
2,origen_dataset,str
3,id_cita,str
4,fecha_cita,object
5,centro_salud,str
6,canal_agendamiento,str
7,grupo_edad,str
8,especialidad,str
9,costo_estimado_usd,float64



PRIMEROS 5 REGISTROS ORIGINALES


,id_raw,lote_carga,origen_dataset,id_cita,fecha_cita,centro_salud,canal_agendamiento,grupo_edad,especialidad,costo_estimado_usd,espera_dias,recordatorios_enviados,citas_previas_perdidas,tipo_cobertura,sistema_origen,no_asistio,consentimiento_contacto,fecha_carga,fecha_ingesta
0,1,20260928T002744032220Z,ampliado,RSV2-000001,2026-04-09,DIRIAMBA,WEB,60+,MEDICINA GENERAL,62.06,33.0,3.0,1.0,MIXTA,HIS,0,SI,2026-04-10,2026-09-28 00:27:45.112971
1,2,20260928T002744032220Z,ampliado,RSV2-000002,2026-04-30,JINOTEPE,VENTANILLA,45-59,LABORATORIO,18.44,6.0,3.0,4.0,PRIVADA,HIS,1,SI,2026-05-01,2026-09-28 00:27:45.112971
2,3,20260928T002744032220Z,ampliado,RSV2-000003,2026-02-27,DIRIAMBA,CALL CENTER,NaN,GINECOLOGÍA,39.48,4.0,3.0,1.0,PÚBLICA,HIS,0,SI,2026-02-28,2026-09-28 00:27:45.112971
3,4,20260928T002744032220Z,ampliado,RSV2-000004,2026-03-28,DIRIAMBA,WEB,30-44,GINECOLOGÍA,26.58,11.0,1.0,0.0,PRIVADA,HIS,0,SI,2026-03-29,2026-09-28 00:27:45.112971
4,5,20260928T002744032220Z,ampliado,RSV2-000005,2026-01-04,DOLORES,VENTANILLA,45-59,GINECOLOGÍA,30.78,10.0,3.0,2.0,PÚBLICA,HIS,NaN,SI,2026-01-05,2026-09-28 00:27:45.112971



CONTROL INICIAL DE IDENTIFICADORES
Registros originales: 10080
Identificadores distintos: 10000
Grupos de IDs repetidos: 80

PASO 1 FINALIZADO
Consultas de solo lectura.
No se modificaron datos de SQL Server.



## PASO 2. Análisis Exploratorio de Datos (EDA)

### Objetivo
Analizar la estructura y calidad inicial de los registros de citas médicas, identificando valores faltantes, duplicidad y posibles anomalías.

### Alcance
El análisis utiliza los 10,080 registros de la tabla dbo.raw_cita.

### Evidencia
Se generan tablas y archivos CSV con los resultados del diagnóstico.

**Importante:** este proceso no modifica los datos originales.


In [2]:

# ============================================================
# OPTATIVO III - GESTIÓN DE LA CALIDAD DE LOS PROCESOS
# PASO 2. ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ============================================================

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

# ============================================================
# 1. PREPARAR LOS DATOS Y LA CARPETA DE EVIDENCIAS
# ============================================================

# Utilizamos los registros consultados en el PASO 1
df_eda = datos_originales.copy()

carpeta_eda = (
    proyecto / "resultados_jupyter" / "optativo_III"
)

carpeta_eda.mkdir(
    parents=True,
    exist_ok=True
)

fecha_ejecucion = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

total_registros = len(df_eda)

print("=" * 65)
print("PASO 2. ANÁLISIS EXPLORATORIO DE DATOS")
print("=" * 65)

print("Registros analizados:", total_registros)
print("Columnas:", len(df_eda.columns))


# ============================================================
# 2. PERFIL DE CALIDAD POR COLUMNA
# ============================================================

perfil_campos = []

for campo in df_eda.columns:

    serie = df_eda[campo]

    # Detectar valores nulos
    faltantes = serie.isna()

    # Detectar cadenas vacías o espacios
    if (
        pd.api.types.is_object_dtype(serie)
        or pd.api.types.is_string_dtype(serie)
    ):

        vacios = (
            serie.astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
        )

        faltantes = faltantes | vacios

    cantidad_faltantes = int(
        faltantes.sum()
    )

    perfil_campos.append({
        "Campo": campo,
        "Tipo": str(serie.dtype),
        "Registros": total_registros,
        "Valores_faltantes": cantidad_faltantes,
        "Porcentaje_faltante": round(
            cantidad_faltantes / total_registros * 100,
            2
        ),
        "Valores_distintos": int(
            serie.nunique(dropna=True)
        )
    })

tabla_perfil = pd.DataFrame(
    perfil_campos
)

tabla_perfil = tabla_perfil.sort_values(
    "Valores_faltantes",
    ascending=False
).reset_index(drop=True)

print("\n1. PERFIL DE COMPLETITUD POR CAMPO")

display(tabla_perfil)


# ============================================================
# 3. ESTADÍSTICAS DE VARIABLES NUMÉRICAS
# ============================================================

columnas_numericas = [
    "costo_estimado_usd",
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas"
]

datos_numericos = df_eda[
    columnas_numericas
].apply(
    pd.to_numeric,
    errors="coerce"
)

print("\n2. ESTADÍSTICAS DE VARIABLES NUMÉRICAS")

display(
    datos_numericos.describe().T.round(2)
)


# ============================================================
# 4. IDENTIFICADORES REPETIDOS
# ============================================================

ids = (
    df_eda["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

frecuencias = (
    ids.dropna().value_counts()
)

grupos_repetidos = int(
    (frecuencias > 1).sum()
)

filas_con_ids_repetidos = int(
    frecuencias[frecuencias > 1].sum()
)

print("\n3. ANÁLISIS DE IDENTIFICADORES")

print(
    "Identificadores distintos:",
    ids.nunique()
)

print(
    "Grupos de IDs repetidos:",
    grupos_repetidos
)

print(
    "Filas involucradas:",
    filas_con_ids_repetidos
)


# ============================================================
# 5. IDENTIFICAR FILAS COMPLETAMENTE DUPLICADAS
# ============================================================

# Utilizamos las 15 columnas del dataset original.
# No incluimos columnas técnicas de SQL Server.

columnas_cita = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "canal_agendamiento",
    "grupo_edad",
    "especialidad",
    "costo_estimado_usd",
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas",
    "tipo_cobertura",
    "sistema_origen",
    "no_asistio",
    "consentimiento_contacto",
    "fecha_carga"
]

faltan_columnas = (
    set(columnas_cita) - set(df_eda.columns)
)

if faltan_columnas:
    raise ValueError(
        f"Faltan columnas: {sorted(faltan_columnas)}"
    )

mascara_duplicadas = df_eda.duplicated(
    subset=columnas_cita,
    keep="first"
)

filas_duplicadas = int(
    mascara_duplicadas.sum()
)

print("\n4. ANÁLISIS DE FILAS DUPLICADAS")

print(
    "Filas duplicadas adicionales:",
    filas_duplicadas
)


# ============================================================
# 6. IDENTIFICAR POSIBLES VALORES ANÓMALOS
# ============================================================

reglas_numericas = [
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas"
]

resultados_anomalias = []

for campo in reglas_numericas:

    serie = pd.to_numeric(
        df_eda[campo],
        errors="coerce"
    )

    # Valores numéricos negativos
    negativos = int(
        serie.lt(0).sum()
    )

    # Valores numéricos no enteros
    no_enteros = int(
        (serie.notna() & serie.ne(serie.round())).sum()
    )

    resultados_anomalias.append({
        "Campo": campo,
        "Negativos": negativos,
        "No_enteros": no_enteros
    })

tabla_anomalias = pd.DataFrame(
    resultados_anomalias
)

print("\n5. POSIBLES ANOMALÍAS NUMÉRICAS")

display(tabla_anomalias)


# ============================================================
# 7. EXPLORAR LA VARIABLE OBJETIVO
# ============================================================

print("\n6. DISTRIBUCIÓN DE NO_ASISTIO")

distribucion_asistencia = (
    df_eda["no_asistio"]
    .astype("string")
    .str.strip()
    .fillna("SIN DATO")
    .replace("", "SIN DATO")
    .value_counts()
    .rename_axis("Valor")
    .reset_index(name="Cantidad")
)

display(distribucion_asistencia)


# ============================================================
# 8. RESUMEN GENERAL DEL EDA
# ============================================================

resumen_eda = pd.DataFrame([
    {
        "Comprobación": "Registros originales",
        "Resultado": total_registros
    },
    {
        "Comprobación": "Columnas",
        "Resultado": len(df_eda.columns)
    },
    {
        "Comprobación": "Identificadores distintos",
        "Resultado": int(ids.nunique())
    },
    {
        "Comprobación": "Grupos de IDs repetidos",
        "Resultado": grupos_repetidos
    },
    {
        "Comprobación": "Filas con IDs repetidos",
        "Resultado": filas_con_ids_repetidos
    },
    {
        "Comprobación": "Filas duplicadas adicionales",
        "Resultado": filas_duplicadas
    }
])

print("\n" + "=" * 65)
print("RESUMEN DEL DIAGNÓSTICO EXPLORATORIO")
print("=" * 65)

display(resumen_eda)


# ============================================================
# 9. GUARDAR EVIDENCIAS
# ============================================================

tabla_perfil.to_csv(
    carpeta_eda / f"eda_perfil_{fecha_ejecucion}.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_anomalias.to_csv(
    carpeta_eda / f"eda_anomalias_{fecha_ejecucion}.csv",
    index=False,
    encoding="utf-8-sig"
)

distribucion_asistencia.to_csv(
    carpeta_eda / f"eda_asistencia_{fecha_ejecucion}.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_eda.to_csv(
    carpeta_eda / f"eda_resumen_{fecha_ejecucion}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nEVIDENCIAS GUARDADAS")
print("Carpeta:", carpeta_eda)
print("- Perfil de completitud")
print("- Análisis de anomalías")
print("- Distribución de asistencia")
print("- Resumen del EDA")

print("\nPASO 2 FINALIZADO")
print("Los datos originales no fueron modificados.")


PASO 2. ANÁLISIS EXPLORATORIO DE DATOS
Registros analizados: 10080
Columnas: 19

1. PERFIL DE COMPLETITUD POR CAMPO


,Campo,Tipo,Registros,Valores_faltantes,Porcentaje_faltante,Valores_distintos
0,especialidad,str,10080,120,1.19,10
1,centro_salud,str,10080,120,1.19,12
2,canal_agendamiento,str,10080,120,1.19,8
3,espera_dias,float64,10080,120,1.19,62
4,recordatorios_enviados,float64,10080,120,1.19,5
5,citas_previas_perdidas,float64,10080,120,1.19,6
6,grupo_edad,str,10080,120,1.19,4
7,no_asistio,str,10080,80,0.79,2
8,origen_dataset,str,10080,0,0.00,1
9,id_raw,int64,10080,0,0.00,10080



2. ESTADÍSTICAS DE VARIABLES NUMÉRICAS


,count,mean,std,min,25%,50%,75%,max
costo_estimado_usd,10080.0,29.07,12.67,-1.0,20.42,29.1,37.65,74.09
espera_dias,9960.0,19.88,16.15,-1.0,7.00,15.0,29.00,60.00
recordatorios_enviados,9960.0,1.13,1.09,-1.0,0.00,1.0,2.00,3.00
citas_previas_perdidas,9960.0,1.41,1.27,-1.0,0.00,1.0,2.00,4.00



3. ANÁLISIS DE IDENTIFICADORES
Identificadores distintos: 10000
Grupos de IDs repetidos: 80
Filas involucradas: 160

4. ANÁLISIS DE FILAS DUPLICADAS
Filas duplicadas adicionales: 47

5. POSIBLES ANOMALÍAS NUMÉRICAS


...


6. DISTRIBUCIÓN DE NO_ASISTIO


,Valor,Cantidad
0,0,6263
1,1,3737
2,SIN DATO,80



RESUMEN DEL DIAGNÓSTICO EXPLORATORIO


,Comprobación,Resultado
0,Registros originales,10080
1,Columnas,19
2,Identificadores distintos,10000
3,Grupos de IDs repetidos,80
4,Filas con IDs repetidos,160
5,Filas duplicadas adicionales,47



EVIDENCIAS GUARDADAS
Carpeta: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_III
- Perfil de completitud
- Análisis de anomalías
- Distribución de asistencia
- Resumen del EDA

PASO 2 FINALIZADO
Los datos originales no fueron modificados.


In [3]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO III - GESTIÓN DE LA CALIDAD DE LOS PROCESOS
#
# PASO 3. AUDITORÍA INTEGRAL DE CALIDAD DE DATOS
#
# POL-02: COMPLETITUD
# POL-03: ESTANDARIZACIÓN
# POL-04: DUPLICIDAD
# POL-05: VALORES ANÓMALOS
#
# EVIDENCIAS GUARDADAS EN CARPETA EXCLUSIVA
# ============================================================

import json
import re
import unicodedata
import pandas as pd

from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display


# ============================================================
# 1. PREPARAR DATOS Y CARPETA DE AUDITORÍA
# ============================================================

# Datos originales cargados en el PASO 1
df = datos_originales.copy()

# Carpeta exclusiva del PASO 3
carpeta_auditoria = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "auditoria_integral"
)

carpeta_auditoria.mkdir(
    parents=True,
    exist_ok=True
)

# Fecha para identificar la ejecución
fecha = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

total = len(df)

if total == 0:
    raise ValueError(
        "No existen registros para auditar."
    )

if not df["id_raw"].is_unique:
    raise ValueError(
        "El identificador id_raw no es único."
    )

hallazgos = []

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("PASO 3. AUDITORÍA INTEGRAL DE CALIDAD")
print("=" * 70)

print("\nRegistros originales:", total)
print("Columnas:", len(df.columns))
print("Fuente: dbo.raw_cita")
print("Carpeta de evidencias:", carpeta_auditoria)


# ============================================================
# 2. FUNCIONES GENERALES DE AUDITORÍA
# ============================================================

def es_vacio(valor):
    """
    Detecta valores nulos, vacíos o con espacios.
    """

    if pd.isna(valor):
        return True

    if isinstance(valor, str):
        return not valor.strip()

    return False


def mostrar_valor(valor):
    """
    Presenta el valor encontrado en el dato original.
    """

    if pd.isna(valor):
        return "<NULO>"

    if isinstance(valor, str) and not valor.strip():
        return "<VACÍO>"

    return str(valor)


def normalizar_texto(valor):
    """
    Normaliza texto para comparar categorías.
    No modifica la base de datos.
    """

    if es_vacio(valor):
        return ""

    texto = str(valor).strip().upper()

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    texto = "".join(
        caracter
        for caracter in unicodedata.normalize(
            "NFD",
            texto
        )
        if unicodedata.category(caracter) != "Mn"
    )

    return texto


def registrar_hallazgo(
    fila,
    politica,
    tipo,
    campo,
    valor,
    regla,
    tratamiento
):
    """
    Registra una incidencia individual con trazabilidad.
    """

    hallazgos.append({
        "id_raw": fila["id_raw"],
        "id_cita": fila["id_cita"],
        "Politica": politica,
        "Tipo_hallazgo": tipo,
        "Campo": campo,
        "Valor_original": mostrar_valor(valor),
        "Regla_incumplida": regla,
        "Tratamiento_propuesto": tratamiento,
        "Estado": "DETECTADO",
        "Decision_final": "PENDIENTE"
    })


# ============================================================
# 3. AUDITORÍA POL-02: COMPLETITUD
# ============================================================

print("\n" + "=" * 70)
print("POL-02. POLÍTICA DE COMPLETITUD")
print("=" * 70)

campos_obligatorios = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad",
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados",
    "no_asistio"
]

faltan_columnas = (
    set(campos_obligatorios) - set(df.columns)
)

if faltan_columnas:
    raise ValueError(
        f"Faltan columnas: {sorted(faltan_columnas)}"
    )

mascara_faltantes = pd.DataFrame(
    index=df.index
)

for campo in campos_obligatorios:

    mascara = df[campo].map(es_vacio)

    mascara_faltantes[campo] = mascara

    for indice in df.index[mascara]:

        fila = df.loc[indice]

        registrar_hallazgo(
            fila=fila,
            politica="POL-02",
            tipo="CAMPO_OBLIGATORIO_FALTANTE",
            campo=campo,
            valor=fila[campo],
            regla="El campo obligatorio debe contener información",
            tratamiento="Verificar el dato en una fuente confiable"
        )

# Calcular registros incompletos
registros_incompletos = int(
    mascara_faltantes.any(axis=1).sum()
)

registros_completos = total - registros_incompletos

tasa_integridad = (
    registros_completos / total
) * 100

print("Registros completos:", registros_completos)
print("Registros incompletos:", registros_incompletos)
print(f"Tasa de integridad: {tasa_integridad:.2f}%")


# ============================================================
# 4. AUDITORÍA POL-03: ESTANDARIZACIÓN
# ============================================================

print("\n" + "=" * 70)
print("POL-03. POLÍTICA DE ESTANDARIZACIÓN")
print("=" * 70)

# Catálogos de referencia provisionales.
# Se deben confirmar con el responsable institucional.

catalogos_referencia = {
    "centro_salud": [
        "JINOTEPE",
        "DIRIAMBA",
        "SAN MARCOS",
        "DOLORES"
    ],
    "especialidad": [
        "MEDICINA GENERAL",
        "LABORATORIO",
        "GINECOLOGÍA",
        "PEDIATRÍA",
        "CARDIOLOGÍA"
    ],
    "canal_agendamiento": [
        "CALL CENTER",
        "WEB",
        "VENTANILLA",
        "REFERENCIA MÉDICA"
    ]
}

# Recuperar catálogos del proyecto si están
# disponibles como listas por campo.

ruta_catalogos = (
    proyecto / "catalogos_propuestos_gobernanza.json"
)

fuente_catalogos = "Catálogos provisionales"

if ruta_catalogos.exists():

    with open(
        ruta_catalogos,
        encoding="utf-8"
    ) as archivo:

        contenido = json.load(archivo)

    if isinstance(contenido, dict):

        nodo = contenido.get(
            "catalogos",
            contenido
        )

        if isinstance(nodo, dict):

            for campo in catalogos_referencia:

                valores = nodo.get(campo)

                if (
                    isinstance(valores, list)
                    and all(
                        isinstance(valor, str)
                        for valor in valores
                    )
                ):

                    catalogos_referencia[campo] = valores

                    fuente_catalogos = (
                        "Archivo JSON y catálogos provisionales"
                    )

for campo, valores_permitidos in catalogos_referencia.items():

    equivalencias = {
        normalizar_texto(valor): valor
        for valor in valores_permitidos
    }

    for _, fila in df.iterrows():

        original = fila[campo]

        # Los vacíos se auditan mediante POL-02
        if es_vacio(original):
            continue

        clave_normalizada = normalizar_texto(
            original
        )

        # Categoría no incluida en el catálogo
        if clave_normalizada not in equivalencias:

            registrar_hallazgo(
                fila=fila,
                politica="POL-03",
                tipo="FUERA_DE_CATALOGO",
                campo=campo,
                valor=original,
                regla="Debe pertenecer al catálogo de referencia",
                tratamiento="Revisar el valor con el responsable del catálogo"
            )

        else:

            valor_estandar = equivalencias[
                clave_normalizada
            ]

            # Detectar diferencias de escritura
            if str(original) != valor_estandar:

                registrar_hallazgo(
                    fila=fila,
                    politica="POL-03",
                    tipo="FORMATO_NO_ESTANDAR",
                    campo=campo,
                    valor=original,
                    regla=f"Formato esperado: {valor_estandar}",
                    tratamiento="Normalizar formato tras verificar la equivalencia"
                )

print("Referencia utilizada:", fuente_catalogos)
print("Evaluación de categorías finalizada.")


# ============================================================
# 5. AUDITORÍA POL-04: CONTROL DE DUPLICIDAD
# ============================================================

print("\n" + "=" * 70)
print("POL-04. POLÍTICA DE CONTROL DE DUPLICIDAD")
print("=" * 70)

# ------------------------------------------------------------
# A. IDENTIFICADORES DE CITA REPETIDOS
# ------------------------------------------------------------

ids = (
    df["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

frecuencias = ids.dropna().value_counts()

ids_repetidos = frecuencias[
    frecuencias > 1
].index

mascara_ids_repetidos = ids.isin(
    ids_repetidos
)

for indice in df.index[mascara_ids_repetidos]:

    fila = df.loc[indice]

    registrar_hallazgo(
        fila=fila,
        politica="POL-04",
        tipo="IDENTIFICADOR_REPETIDO",
        campo="id_cita",
        valor=fila["id_cita"],
        regla="Cada identificador de cita debe ser único",
        tratamiento="Revisar registros del mismo identificador"
    )

grupos_ids_repetidos = len(ids_repetidos)

filas_ids_repetidos = int(
    mascara_ids_repetidos.sum()
)

# ------------------------------------------------------------
# B. FILAS COMPLETAMENTE DUPLICADAS
# ------------------------------------------------------------

# Se comparan las columnas originales de las citas.
# Las columnas técnicas de SQL Server no se incluyen.

columnas_originales = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "canal_agendamiento",
    "grupo_edad",
    "especialidad",
    "costo_estimado_usd",
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas",
    "tipo_cobertura",
    "sistema_origen",
    "no_asistio",
    "consentimiento_contacto",
    "fecha_carga"
]

columnas_ausentes = (
    set(columnas_originales) - set(df.columns)
)

if columnas_ausentes:
    raise ValueError(
        f"Faltan columnas originales: {sorted(columnas_ausentes)}"
    )

# Identificar todas las filas involucradas
mascara_grupo_duplicado = df.duplicated(
    subset=columnas_originales,
    keep=False
)

# Identificar solo las copias adicionales
mascara_duplicadas_adicionales = df.duplicated(
    subset=columnas_originales,
    keep="first"
)

filas_duplicadas = int(
    mascara_duplicadas_adicionales.sum()
)

for indice in df.index[mascara_grupo_duplicado]:

    fila = df.loc[indice]

    registrar_hallazgo(
        fila=fila,
        politica="POL-04",
        tipo="FILA_COMPLETAMENTE_DUPLICADA",
        campo="FILA_COMPLETA",
        valor=fila["id_cita"],
        regla="La combinación de campos originales no debe repetirse",
        tratamiento="Verificar qué registro debe conservarse"
    )

tasa_duplicidad_ids = (
    grupos_ids_repetidos / total
) * 100

porcentaje_filas_duplicadas = (
    filas_duplicadas / total
) * 100

print("Grupos de IDs repetidos:", grupos_ids_repetidos)
print("Filas involucradas:", filas_ids_repetidos)
print("Filas duplicadas adicionales:", filas_duplicadas)

print(
    f"Tasa de duplicidad de identificadores: "
    f"{tasa_duplicidad_ids:.2f}%"
)

print(
    f"Porcentaje de filas duplicadas: "
    f"{porcentaje_filas_duplicadas:.2f}%"
)


# ============================================================
# 6. AUDITORÍA POL-05: VALORES ANÓMALOS
# ============================================================

print("\n" + "=" * 70)
print("POL-05. POLÍTICA DE CONTROL DE VALORES ANÓMALOS")
print("=" * 70)

campos_numericos = [
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas"
]

for campo in campos_numericos:

    for _, fila in df.iterrows():

        valor = fila[campo]

        if es_vacio(valor):
            # Los valores faltantes se registran en POL-02
            continue

        numero = pd.to_numeric(
            pd.Series([valor]),
            errors="coerce"
        ).iloc[0]

        if pd.isna(numero):

            registrar_hallazgo(
                fila=fila,
                politica="POL-05",
                tipo="NUMERO_INVALIDO",
                campo=campo,
                valor=valor,
                regla="Debe ser un número entero no negativo",
                tratamiento="Consultar el valor verdadero en la fuente"
            )

        elif numero < 0 or numero != round(numero):

            registrar_hallazgo(
                fila=fila,
                politica="POL-05",
                tipo="VALOR_NUMERICO_INVALIDO",
                campo=campo,
                valor=valor,
                regla="Debe ser un entero mayor o igual a cero",
                tratamiento="Verificar el dato antes de corregirlo"
            )

# Validación de la variable objetivo no_asistio

for _, fila in df.iterrows():

    valor = fila["no_asistio"]

    if es_vacio(valor):
        continue

    if str(valor).strip() not in ["0", "1"]:

        registrar_hallazgo(
            fila=fila,
            politica="POL-05",
            tipo="RESULTADO_INVALIDO",
            campo="no_asistio",
            valor=valor,
            regla="Solo se permiten los valores 0 y 1",
            tratamiento="Comprobar el resultado real de la cita"
        )

print("Validación de valores anómalos finalizada.")


# ============================================================
# 7. ALERTAS ADICIONALES DEL EDA
# ============================================================

# El EDA detectó un costo estimado negativo.
# Lo registramos como observación adicional y
# no como parte de una política no definida.

costos = pd.to_numeric(
    df["costo_estimado_usd"],
    errors="coerce"
)

for indice in df.index[costos.lt(0)]:

    fila = df.loc[indice]

    registrar_hallazgo(
        fila=fila,
        politica="EDA-REVISION",
        tipo="COSTO_NEGATIVO",
        campo="costo_estimado_usd",
        valor=fila["costo_estimado_usd"],
        regla="Valor negativo que requiere revisión",
        tratamiento="Consultar la regla de negocio y el dato original"
    )


# ============================================================
# 8. CONSTRUIR AUDITORÍA GENERAL
# ============================================================

auditoria_general = pd.DataFrame(
    hallazgos
)

# Resumen por política
resumen_politicas = (
    auditoria_general
    .groupby("Politica")
    .agg(
        Incidencias=("id_raw", "size"),
        Registros_afectados=("id_raw", "nunique")
    )
    .reset_index()
)

# Incluir políticas sin hallazgos
politicas_esperadas = [
    "POL-02",
    "POL-03",
    "POL-04",
    "POL-05",
    "EDA-REVISION"
]

resumen_politicas = (
    resumen_politicas
    .set_index("Politica")
    .reindex(
        politicas_esperadas,
        fill_value=0
    )
    .reset_index()
)

# Resumen por tipo de hallazgo
resumen_tipos = (
    auditoria_general
    .groupby(["Politica", "Tipo_hallazgo"])
    .agg(
        Incidencias=("id_raw", "size"),
        Registros_afectados=("id_raw", "nunique")
    )
    .reset_index()
)

# Agrupar los hallazgos por registro original
registros_afectados = (
    auditoria_general
    .groupby(
        ["id_raw", "id_cita"],
        dropna=False
    )
    .agg(
        Politicas=(
            "Politica",
            lambda valores: ", ".join(
                sorted(set(valores))
            )
        ),
        Tipos_hallazgo=(
            "Tipo_hallazgo",
            lambda valores: ", ".join(
                sorted(set(valores))
            )
        ),
        Total_incidencias=("Politica", "size")
    )
    .reset_index()
)


# ============================================================
# 9. BITÁCORA INICIAL DE TRATAMIENTO
# ============================================================

bitacora = auditoria_general.copy()

bitacora["Fuente"] = "dbo.raw_cita"

bitacora["Fecha_deteccion_UTC"] = (
    datetime.now(timezone.utc).isoformat()
)

bitacora["Valor_despues"] = pd.NA

bitacora["Accion_ejecutada"] = pd.NA

bitacora["Fecha_correccion_UTC"] = pd.NA

bitacora["Verificacion_posterior"] = (
    "PENDIENTE"
)

# IMPORTANTE:
# La bitácora registra los tratamientos propuestos.
# No se atribuyen correcciones que todavía no se
# han aplicado sobre un conjunto depurado.


# ============================================================
# 10. CALCULAR LOS TRES KPI DEL II CORTE
# ============================================================

tabla_kpis = pd.DataFrame([
    {
        "Código": "IND-01",
        "KPI": "Tasa de integridad de registros de citas",
        "Resultado (%)": round(tasa_integridad, 2),
        "Meta": ">98%",
        "Estado": (
            "CUMPLE"
            if tasa_integridad > 98
            else "NO CUMPLE"
        )
    },
    {
        "Código": "IND-04",
        "KPI": "Porcentaje de filas duplicadas",
        "Resultado (%)": round(
            porcentaje_filas_duplicadas, 2
        ),
        "Meta": "0%",
        "Estado": (
            "CUMPLE"
            if filas_duplicadas == 0
            else "NO CUMPLE"
        )
    },
    {
        "Código": "IND-05",
        "KPI": "Tasa de duplicidad de identificadores de cita",
        "Resultado (%)": round(
            tasa_duplicidad_ids, 2
        ),
        "Meta": "0%",
        "Estado": (
            "CUMPLE"
            if grupos_ids_repetidos == 0
            else "NO CUMPLE"
        )
    }
])


# ============================================================
# 11. MOSTRAR RESULTADOS DE AUDITORÍA
# ============================================================

print("\n" + "=" * 70)
print("RESULTADOS DE LA AUDITORÍA INTEGRAL")
print("=" * 70)

print("\n1. RESUMEN POR POLÍTICA")
display(resumen_politicas)

print("\n2. DETALLE POR TIPO DE HALLAZGO")
display(resumen_tipos)

print("\n3. REGISTROS AFECTADOS")
display(registros_afectados.head(15))

print("\n4. INCIDENCIAS INDIVIDUALES")
display(auditoria_general.head(15))

print("\n5. TRES KPI DEL II CORTE")
display(tabla_kpis)

print("\n6. RESUMEN DE DUPLICIDAD")

print(
    "Grupos de identificadores repetidos:",
    grupos_ids_repetidos
)

print(
    "Filas involucradas:",
    filas_ids_repetidos
)

print(
    "Filas duplicadas adicionales:",
    filas_duplicadas
)

print("\n7. POLÍTICA NO EVALUABLE")

print(
    "POL-01: No es posible verificar el plazo "
    "de cierre de 24 horas sin disponer de "
    "la fecha de cierre de las citas."
)


# ============================================================
# 12. VERIFICAR CONSISTENCIA DE RESULTADOS
# ============================================================

ids_pol02 = (
    auditoria_general.loc[
        auditoria_general["Politica"] == "POL-02",
        "id_raw"
    ].nunique()
)

assert (
    registros_incompletos == ids_pol02
), "No coinciden los registros incompletos."

assert (
    grupos_ids_repetidos
    == int((frecuencias > 1).sum())
), "No coinciden los identificadores repetidos."

assert (
    filas_duplicadas
    == int(mascara_duplicadas_adicionales.sum())
), "No coinciden las filas duplicadas."

print("\nCONTROLES DE CONSISTENCIA: SUPERADOS")


# ============================================================
# 13. GUARDAR EVIDENCIAS EN CARPETA EXCLUSIVA
# ============================================================

print("\n" + "=" * 70)
print("GUARDANDO EVIDENCIAS DE AUDITORÍA")
print("=" * 70)

# A. Archivos generales
archivos_generales = {
    "auditoria_general": auditoria_general,
    "auditoria_resumen_politicas": resumen_politicas,
    "auditoria_tipos_hallazgo": resumen_tipos,
    "auditoria_registros_afectados": registros_afectados,
    "bitacora_tratamiento_inicial": bitacora,
    "kpi_calidad_diagnostico": tabla_kpis
}

for nombre, tabla in archivos_generales.items():

    ruta = (
        carpeta_auditoria
        / f"{nombre}_{fecha}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)


# ============================================================
# 14. GUARDAR AUDITORÍA INDIVIDUAL POR POLÍTICA
# ============================================================

# Estos archivos permiten mostrar al profesor
# todos los registros afectados por cada política.

nombres_politicas = {
    "POL-02": "POL02_completitud",
    "POL-03": "POL03_estandarizacion",
    "POL-04": "POL04_duplicidad",
    "POL-05": "POL05_valores_anomalos"
}

for codigo, nombre_archivo in nombres_politicas.items():

    detalle = auditoria_general.loc[
        auditoria_general["Politica"] == codigo
    ].copy()

    ruta = (
        carpeta_auditoria
        / f"{nombre_archivo}_{fecha}.csv"
    )

    detalle.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Guardado:",
        ruta.name,
        "| Incidencias:",
        len(detalle)
    )


# ============================================================
# 15. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 70)
print("PASO 3. AUDITORÍA INTEGRAL FINALIZADA")
print("=" * 70)

print("\nRegistros originales:", total)

print(
    "Registros con al menos un hallazgo:",
    registros_afectados["id_raw"].nunique()
)

print(
    "Total de incidencias registradas:",
    len(auditoria_general)
)

print("\nCarpeta de evidencias:")
print(carpeta_auditoria)

print("\nArchivos generales:", len(archivos_generales))
print("Archivos individuales por política:", len(nombres_politicas))

print("\nNo se modificaron datos originales.")
print("Los tratamientos correctivos siguen pendientes.")


RED SALUD INTEGRAL CARAZO
PASO 3. AUDITORÍA INTEGRAL DE CALIDAD

Registros originales: 10080
Columnas: 19
Fuente: dbo.raw_cita
Carpeta de evidencias: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_III\auditoria_integral

POL-02. POLÍTICA DE COMPLETITUD
Registros completos: 9160
Registros incompletos: 920
Tasa de integridad: 90.87%

POL-03. POLÍTICA DE ESTANDARIZACIÓN
Referencia utilizada: Archivo JSON y catálogos provisionales
Evaluación de categorías finalizada.

POL-04. POLÍTICA DE CONTROL DE DUPLICIDAD
Grupos de IDs repetidos: 80
Filas involucradas: 160
Filas duplicadas adicionales: 47
Tasa de duplicidad de identificadores: 0.79%
Porcentaje de filas duplicadas: 0.47%

POL-05. POLÍTICA DE CONTROL DE VALORES ANÓMALOS
Validación de valores anómalos finalizada.

RESULTADOS DE LA AUDITORÍA INTEGRAL

1. RESUMEN POR POLÍTICA


,Politica,Incidencias,Registros_afectados
0,POL-02,920,920
1,POL-03,273,273
2,POL-04,254,160
3,POL-05,3,3
4,EDA-REVISION,1,1



2. DETALLE POR TIPO DE HALLAZGO


,Politica,Tipo_hallazgo,Incidencias,Registros_afectados
0,EDA-REVISION,COSTO_NEGATIVO,1,1
1,POL-02,CAMPO_OBLIGATORIO_FALTANTE,920,920
2,POL-03,FORMATO_NO_ESTANDAR,273,273
3,POL-04,FILA_COMPLETAMENTE_DUPLICADA,94,94
4,POL-04,IDENTIFICADOR_REPETIDO,160,160
5,POL-05,VALOR_NUMERICO_INVALIDO,3,3



3. REGISTROS AFECTADOS


...


4. INCIDENCIAS INDIVIDUALES


,id_raw,id_cita,Politica,Tipo_hallazgo,Campo,Valor_original,Regla_incumplida,Tratamiento_propuesto,Estado,Decision_final
0,38,RSV2-000038,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
1,330,RSV2-000330,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
2,511,RSV2-000511,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
3,556,RSV2-000556,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
4,673,RSV2-000673,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
5,675,RSV2-000675,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
6,680,RSV2-000680,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
7,683,RSV2-000683,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
8,766,RSV2-000766,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE
9,795,RSV2-000795,POL-02,CAMPO_OBLIGATORIO_FALTANTE,centro_salud,<NULO>,El campo obligatorio debe contener información,Verificar el dato en una fuente confiable,DETECTADO,PENDIENTE



5. TRES KPI DEL II CORTE


,Código,KPI,Resultado (%),Meta,Estado
0,IND-01,Tasa de integridad de registros de citas,90.87,>98%,NO CUMPLE
1,IND-04,Porcentaje de filas duplicadas,0.47,0%,NO CUMPLE
2,IND-05,Tasa de duplicidad de identificadores de cita,0.79,0%,NO CUMPLE



6. RESUMEN DE DUPLICIDAD
Grupos de identificadores repetidos: 80
Filas involucradas: 160
Filas duplicadas adicionales: 47

7. POLÍTICA NO EVALUABLE
POL-01: No es posible verificar el plazo de cierre de 24 horas sin disponer de la fecha de cierre de las citas.

CONTROLES DE CONSISTENCIA: SUPERADOS

GUARDANDO EVIDENCIAS DE AUDITORÍA
Guardado: auditoria_general_20261024T210501Z.csv
Guardado: auditoria_resumen_politicas_20261024T210501Z.csv
Guardado: auditoria_tipos_hallazgo_20261024T210501Z.csv
Guardado: auditoria_registros_afectados_20261024T210501Z.csv
Guardado: bitacora_tratamiento_inicial_20261024T210501Z.csv
Guardado: kpi_calidad_diagnostico_20261024T210501Z.csv
Guardado: POL02_completitud_20261024T210501Z.csv | Incidencias: 920
Guardado: POL03_estandarizacion_20261024T210501Z.csv | Incidencias: 273
Guardado: POL04_duplicidad_20261024T210501Z.csv | Incidencias: 254
Guardado: POL05_valores_anomalos_20261024T210501Z.csv | Incidencias: 3

PASO 3. AUDITORÍA INTEGRAL FINALIZADA

Registro


## PASO 4. Limpieza de datos y trazabilidad de tratamientos

### Objetivo
Aplicar tratamientos controlados sobre una copia de los datos originales, documentar las decisiones tomadas y verificar la calidad del conjunto resultante.

### Criterios
- Conservar los registros originales sin modificaciones.
- Normalizar categorías con equivalencias verificables.
- Retirar copias exactamente duplicadas del conjunto apto.
- Separar los registros incompletos, inconsistentes o no verificables.
- Documentar las decisiones por registro.
- Calcular los KPI antes y después del tratamiento.
- Medir la cantidad de información conservada.

La exclusión de un registro del conjunto apto no significa que el registro original haya sido eliminado de SQL Server.


In [4]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO III - GESTIÓN DE LA CALIDAD DE LOS PROCESOS
#
# PASO 4. LIMPIEZA Y TRAZABILIDAD DE TRATAMIENTOS
# ============================================================

import re
import unicodedata
import pandas as pd

from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. PREPARAR DATOS Y DIRECTORIO
# ============================================================

original = datos_originales.copy()

# Orden determinista para conservar copias originales
trabajo = original.sort_values(
    "id_raw",
    kind="stable"
).copy()

carpeta_limpieza = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "limpieza"
)

carpeta_limpieza.mkdir(
    parents=True,
    exist_ok=True
)

fecha_limpieza = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

total_original = len(original)

if total_original == 0:
    raise ValueError("No existen registros originales.")

if not original["id_raw"].is_unique:
    raise ValueError("id_raw debe ser único.")

# Comprobar que se ejecutó la configuración de auditoría
if "catalogos_referencia" not in globals():
    raise RuntimeError(
        "Ejecuta primero el PASO 3 para cargar "
        "los catálogos de referencia."
    )


# ============================================================
# 2. DEFINIR LAS REGLAS DE CALIDAD
# ============================================================

campos_obligatorios = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad",
    "espera_dias",
    "citas_previas_perdidas",
    "recordatorios_enviados",
    "no_asistio"
]

columnas_originales = [
    "id_cita",
    "fecha_cita",
    "centro_salud",
    "canal_agendamiento",
    "grupo_edad",
    "especialidad",
    "costo_estimado_usd",
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas",
    "tipo_cobertura",
    "sistema_origen",
    "no_asistio",
    "consentimiento_contacto",
    "fecha_carga"
]

campos_numericos = [
    "espera_dias",
    "recordatorios_enviados",
    "citas_previas_perdidas"
]

columnas_requeridas = set(
    campos_obligatorios
    + columnas_originales
    + ["id_raw"]
)

faltan_columnas = (
    columnas_requeridas - set(trabajo.columns)
)

if faltan_columnas:
    raise ValueError(
        f"Faltan columnas: {sorted(faltan_columnas)}"
    )


# ============================================================
# 3. FUNCIONES DE VALIDACIÓN
# ============================================================

def detectar_faltantes(serie):
    """Detecta nulos, textos vacíos y espacios."""

    return (
        serie.isna()
        | serie.astype("string")
          .str.strip()
          .eq("")
          .fillna(False)
    )


def normalizar_categoria(valor):
    """Normaliza para comparar equivalencias."""

    if pd.isna(valor):
        return ""

    texto = str(valor).strip().upper()

    texto = re.sub(r"\s+", " ", texto)

    texto = "".join(
        caracter
        for caracter in unicodedata.normalize(
            "NFD", texto
        )
        if unicodedata.category(caracter) != "Mn"
    )

    return texto


def calcular_kpi(datos):

    n = len(datos)

    if n == 0:
        raise ValueError(
            "No existen registros para calcular los KPI."
        )

    faltantes = pd.DataFrame({
        campo: detectar_faltantes(datos[campo])
        for campo in campos_obligatorios
    })

    completos = int(
        (~faltantes.any(axis=1)).sum()
    )

    ids = (
        datos["id_cita"]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )

    frecuencia = ids.dropna().value_counts()

    ids_repetidos = int(
        (frecuencia > 1).sum()
    )

    filas_duplicadas = int(
        datos.duplicated(
            subset=columnas_originales,
            keep="first"
        ).sum()
    )

    return {
        "Registros": n,
        "Completos": completos,
        "IDs_repetidos": ids_repetidos,
        "Filas_duplicadas": filas_duplicadas,
        "IND-01": completos / n * 100,
        "IND-04": filas_duplicadas / n * 100,
        "IND-05": ids_repetidos / n * 100
    }


# ============================================================
# 4. CALCULAR CALIDAD ANTES DEL TRATAMIENTO
# ============================================================

kpi_antes = calcular_kpi(original)

print("=" * 70)
print("PASO 4. LIMPIEZA Y CONTROL DE TRAZABILIDAD")
print("=" * 70)

print("\nDATOS ORIGINALES")
print("Registros:", kpi_antes["Registros"])
print("Registros completos:", kpi_antes["Completos"])
print("IDs repetidos:", kpi_antes["IDs_repetidos"])
print("Filas duplicadas:", kpi_antes["Filas_duplicadas"])


# ============================================================
# 5. PREPARAR LA BITÁCORA DE DECISIONES
# ============================================================

# Cada registro original tendrá una decisión.
# Las razones pueden ser múltiples.

razones = {
    int(id_raw): []
    for id_raw in trabajo["id_raw"]
}

def registrar_razon(mascara, motivo):

    for id_raw in trabajo.loc[
        mascara, "id_raw"
    ]:

        identificador = int(id_raw)

        if motivo not in razones[identificador]:
            razones[identificador].append(motivo)


# Historial de cambios verificables
cambios = []


# ============================================================
# 6. NORMALIZAR CATEGORÍAS EN LA COPIA
# ============================================================

# Solo se normalizan equivalencias existentes en
# el catálogo provisional. No se inventan categorías.

catalogos = catalogos_referencia

for campo in [
    "centro_salud",
    "especialidad",
    "canal_agendamiento"
]:

    equivalencias = {
        normalizar_categoria(valor): valor
        for valor in catalogos[campo]
    }

    claves = trabajo[campo].map(
        normalizar_categoria
    )

    valores_canonicos = claves.map(equivalencias)

    vacios = detectar_faltantes(
        trabajo[campo]
    )

    fuera_catalogo = (
        ~vacios
        & ~claves.isin(equivalencias)
    )

    registrar_razon(
        fuera_catalogo,
        f"POL03_FUERA_CATALOGO:{campo}"
    )

    cambios_formato = (
        ~vacios
        & ~fuera_catalogo
        & trabajo[campo]
          .astype("string")
          .ne(valores_canonicos.astype("string"))
          .fillna(False)
    )

    for indice in trabajo.index[cambios_formato]:

        fila = trabajo.loc[indice]

        cambios.append({
            "id_raw": int(fila["id_raw"]),
            "id_cita": fila["id_cita"],
            "Politica": "POL-03",
            "Campo": campo,
            "Valor_antes": fila[campo],
            "Valor_despues": valores_canonicos.loc[indice],
            "Accion": "NORMALIZACION_DE_FORMATO",
            "Referencia": "Catalogo provisional",
            "Fecha_UTC": fecha_limpieza
        })

    # Modificar únicamente la copia en memoria
    trabajo.loc[
        cambios_formato, campo
    ] = valores_canonicos.loc[
        cambios_formato
    ]

detalle_cambios = pd.DataFrame(
    cambios,
    columns=[
        "id_raw", "id_cita", "Politica",
        "Campo", "Valor_antes", "Valor_despues",
        "Accion", "Referencia", "Fecha_UTC"
    ]
)

print("\nNORMALIZACIÓN DE CATEGORÍAS")
print("Valores normalizados:", len(detalle_cambios))


# ============================================================
# 7. EVALUAR COMPLETITUD - POL-02
# ============================================================

faltantes = pd.DataFrame({
    campo: detectar_faltantes(trabajo[campo])
    for campo in campos_obligatorios
})

mascara_incompletos = faltantes.any(axis=1)

registrar_razon(
    mascara_incompletos,
    "POL02_REGISTRO_INCOMPLETO"
)

# Identificar qué campos faltan en cada registro
detalle_campos_faltantes = faltantes.apply(
    lambda fila: ", ".join(
        campo
        for campo in campos_obligatorios
        if fila[campo]
    ),
    axis=1
)


# ============================================================
# 8. EVALUAR VALORES NUMÉRICOS - POL-05
# ============================================================

mascara_anomalias = pd.Series(
    False,
    index=trabajo.index
)

for campo in campos_numericos:

    serie = pd.to_numeric(
        trabajo[campo],
        errors="coerce"
    )

    no_vacio = ~detectar_faltantes(
        trabajo[campo]
    )

    invalido = (
        no_vacio
        & (
            serie.isna()
            | serie.lt(0)
            | serie.mod(1).ne(0)
        )
    ).fillna(False)

    mascara_anomalias = (
        mascara_anomalias | invalido
    )

    registrar_razon(
        invalido,
        f"POL05_VALOR_INVALIDO:{campo}"
    )

# Verificar que la variable objetivo sea 0 o 1
objetivo = (
    trabajo["no_asistio"]
    .astype("string")
    .str.strip()
)

objetivo_invalido = (
    ~detectar_faltantes(trabajo["no_asistio"])
    & ~objetivo.isin(["0", "1"])
).fillna(False)

registrar_razon(
    objetivo_invalido,
    "POL05_NO_ASISTIO_INVALIDO"
)

mascara_anomalias = (
    mascara_anomalias | objetivo_invalido
)


# ============================================================
# 9. REVISIÓN ADICIONAL DEL COSTO ESTIMADO
# ============================================================

costos = pd.to_numeric(
    trabajo["costo_estimado_usd"],
    errors="coerce"
)

mascara_costo_invalido = (
    ~detectar_faltantes(trabajo["costo_estimado_usd"])
    & (
        costos.isna()
        | costos.lt(0)
    )
).fillna(False)

registrar_razon(
    mascara_costo_invalido,
    "EDA_COSTO_REQUIERE_REVISION"
)


# ============================================================
# 10. IDENTIFICAR COPIAS EXACTAMENTE DUPLICADAS
# ============================================================

# Se consideran las 15 columnas de la cita.
# Las columnas técnicas de SQL Server se conservan,
# pero no intervienen en la comparación.

copia_exacta = trabajo.duplicated(
    subset=columnas_originales,
    keep="first"
)

registrar_razon(
    copia_exacta,
    "POL04_COPIA_DUPLICADA"
)

print("\nCONTROL DE DUPLICADOS")
print(
    "Copias idénticas adicionales tras normalización:",
    int(copia_exacta.sum())
)


# ============================================================
# 11. IDENTIFICAR CONFLICTOS ENTRE IDs REPETIDOS
# ============================================================

# Primero se apartan las copias idénticas.
# Si todavía quedan varios registros con el mismo ID,
# no se elige uno arbitrariamente.

sin_copias = trabajo.loc[
    ~copia_exacta
].copy()

ids_sin_copias = (
    sin_copias["id_cita"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

conflictos_sin_copias = (
    ids_sin_copias.notna()
    & ids_sin_copias.duplicated(keep=False)
)

mascara_conflictos = pd.Series(
    False,
    index=trabajo.index
)

mascara_conflictos.loc[
    sin_copias.index
] = conflictos_sin_copias

registrar_razon(
    mascara_conflictos,
    "POL04_ID_REPETIDO_CON_DATOS_DIFERENTES"
)

print(
    "Registros pendientes por conflictos de ID:",
    int(mascara_conflictos.sum())
)


# ============================================================
# 12. CLASIFICAR EL DESTINO DE CADA REGISTRO
# ============================================================

bitacora = trabajo[
    ["id_raw", "id_cita"]
].copy()

bitacora["Fuente"] = "dbo.raw_cita"

bitacora["Motivos"] = (
    bitacora["id_raw"]
    .map(
        lambda x: " | ".join(
            razones[int(x)]
        )
    )
)

bitacora["Campos_faltantes"] = (
    detalle_campos_faltantes.to_numpy()
)

bitacora["Cantidad_motivos"] = (
    bitacora["id_raw"]
    .map(
        lambda x: len(
            razones[int(x)]
        )
    )
)

bitacora["Estado_final"] = "APTO"

# Las copias idénticas adicionales se excluyen
# del conjunto apto.
bitacora.loc[
    copia_exacta.to_numpy(),
    "Estado_final"
] = "EXCLUIDO_COPIA_CONFIRMADA"

# Los demás casos con motivos quedan para revisión.
requiere_revision = (
    bitacora["Cantidad_motivos"].gt(0)
    & ~copia_exacta.to_numpy()
)

bitacora.loc[
    requiere_revision,
    "Estado_final"
] = "EN_CUARENTENA_PARA_REVISION"

bitacora["Fecha_proceso_UTC"] = fecha_limpieza

bitacora["Observacion"] = (
    "Registro original conservado en SQL Server"
)

# El resultado se obtiene a partir de la decisión
# documentada, no de una cifra establecida manualmente.

mascara_apto = (
    bitacora["Estado_final"].eq("APTO")
)

datos_aptos = trabajo.loc[
    mascara_apto.to_numpy()
].copy()

datos_no_aptos = trabajo.loc[
    ~mascara_apto.to_numpy()
].copy()

datos_no_aptos = datos_no_aptos.merge(
    bitacora[
        [
            "id_raw",
            "Motivos",
            "Campos_faltantes",
            "Estado_final"
        ]
    ],
    on="id_raw",
    how="left",
    validate="one_to_one"
)

if datos_aptos.empty:
    raise ValueError(
        "No quedaron registros aptos. "
        "Revisa las reglas antes de continuar."
    )



# ============================================================
# 13. RECALCULAR LOS KPI OFICIALES DEL II CORTE
#
# IND-01: INTEGRIDAD
# IND-02: CATEGORÍAS NO NORMALIZADAS
# IND-04: FILAS DUPLICADAS
# ============================================================

# Mantener la evaluación general del conjunto apto
kpi_despues = calcular_kpi(datos_aptos)


# ============================================================
# IND-02. FUNCIÓN PARA EVALUAR CATEGORÍAS
# ============================================================

def evaluar_categorias_kpi(datos):

    campos_categoria = [
        "centro_salud",
        "especialidad",
        "canal_agendamiento"
    ]

    categorizados = pd.Series(
        False,
        index=datos.index
    )

    no_normalizados = pd.Series(
        False,
        index=datos.index
    )

    for campo in campos_categoria:

        serie = datos[campo].astype("string")

        tiene_valor = (
            ~detectar_faltantes(serie)
        ).fillna(False)

        equivalencias = {
            normalizar_categoria(valor): valor
            for valor in catalogos_referencia[campo]
        }

        claves = serie.map(normalizar_categoria)

        valores_esperados = claves.map(
            equivalencias
        )

        fuera_catalogo = (
            tiene_valor
            & ~claves.isin(equivalencias)
        ).fillna(False)

        formato_diferente = (
            tiene_valor
            & claves.isin(equivalencias)
            & serie.ne(
                valores_esperados.astype("string")
            ).fillna(False)
        )

        incumple = (
            fuera_catalogo | formato_diferente
        ).fillna(False)

        categorizados = (
            categorizados | tiene_valor
        )

        no_normalizados = (
            no_normalizados | incumple
        )

    cantidad_categorizados = int(
        categorizados.sum()
    )

    cantidad_no_normalizados = int(
        no_normalizados.sum()
    )

    if cantidad_categorizados == 0:
        raise ValueError(
            "No hay registros categorizados."
        )

    porcentaje = (
        cantidad_no_normalizados
        / cantidad_categorizados
        * 100
    )

    return {
        "Categorizados": cantidad_categorizados,
        "No_normalizados": cantidad_no_normalizados,
        "Porcentaje": porcentaje
    }


# ============================================================
# CALCULAR IND-02 ANTES Y DESPUÉS
# ============================================================

categorias_antes = evaluar_categorias_kpi(
    original
)

categorias_despues = evaluar_categorias_kpi(
    datos_aptos
)


# ============================================================
# TABLA COMPARATIVA OFICIAL
# ============================================================

comparativo_kpis = pd.DataFrame([
    {
        "Codigo": "IND-01",
        "KPI": "Tasa de integridad",
        "Antes": round(kpi_antes["IND-01"], 2),
        "Despues": round(kpi_despues["IND-01"], 2),
        "Meta": ">98%",
        "Cumple_despues": (
            "SI"
            if kpi_despues["IND-01"] > 98
            else "NO"
        )
    },
    {
        "Codigo": "IND-02",
        "KPI": "Porcentaje de categorías no normalizadas",
        "Antes": round(
            categorias_antes["Porcentaje"], 2
        ),
        "Despues": round(
            categorias_despues["Porcentaje"], 2
        ),
        "Meta": "<1%",
        "Cumple_despues": (
            "SI"
            if categorias_despues["Porcentaje"] < 1
            else "NO"
        )
    },
    {
        "Codigo": "IND-04",
        "KPI": "Porcentaje de filas duplicadas",
        "Antes": round(kpi_antes["IND-04"], 2),
        "Despues": round(kpi_despues["IND-04"], 2),
        "Meta": "0%",
        "Cumple_despues": (
            "SI"
            if kpi_despues["Filas_duplicadas"] == 0
            else "NO"
        )
    }
])


# ============================================================
# VERIFICACIONES DE CONSISTENCIA
# ============================================================

assert set(comparativo_kpis["Codigo"]) == {
    "IND-01",
    "IND-02",
    "IND-04"
}, "Los KPI no corresponden al documento final."

assert (
    categorias_antes["No_normalizados"] == 273
), "Revisar diferencias con la auditoría POL-03."

print("\nKPI OFICIALES DEL II CORTE")
display(comparativo_kpis)

print("\nCONTROL DE CATEGORÍAS")
print(
    "No normalizados antes:",
    categorias_antes["No_normalizados"]
)
print(
    "No normalizados después:",
    categorias_despues["No_normalizados"]
)



# ============================================================
# 14. CALCULAR COBERTURA Y REGISTROS CONSERVADOS
# ============================================================

cantidad_aptos = len(datos_aptos)
cantidad_no_aptos = len(datos_no_aptos)

cobertura = (
    cantidad_aptos / total_original
) * 100

resumen_destinos = (
    bitacora["Estado_final"]
    .value_counts()
    .rename_axis("Estado")
    .reset_index(name="Registros")
)

resumen_destinos["Porcentaje"] = (
    resumen_destinos["Registros"]
    / total_original * 100
).round(2)

# Control de conservación:
# Todos los registros originales deben estar representados.

assert (
    cantidad_aptos + cantidad_no_aptos
    == total_original
), "No coincide el balance de registros."

assert (
    bitacora["id_raw"].nunique()
    == total_original
), "La bitácora no cubre todos los registros."


# ============================================================
# 15. VERIFICAR QUE EL CONJUNTO APTO CUMPLA LAS REGLAS
# ============================================================

# Completitud
assert not any(
    detectar_faltantes(datos_aptos[campo]).any()
    for campo in campos_obligatorios
), "Persisten campos obligatorios faltantes."

# Duplicados
assert (
    kpi_despues["IDs_repetidos"] == 0
), "Persisten identificadores repetidos."

assert (
    kpi_despues["Filas_duplicadas"] == 0
), "Persisten filas duplicadas."

# Catálogos
for campo in catalogos:

    permitidos = {
        normalizar_categoria(valor)
        for valor in catalogos[campo]
    }

    assert datos_aptos[campo].map(
        normalizar_categoria
    ).isin(permitidos).all(), (
        f"Existen valores fuera del catálogo: {campo}"
    )

# Valores numéricos
for campo in campos_numericos:

    valores = pd.to_numeric(
        datos_aptos[campo],
        errors="coerce"
    )

    assert valores.notna().all(), (
        f"Existen valores no numéricos: {campo}"
    )

    assert (
        valores.ge(0) & valores.mod(1).eq(0)
    ).all(), (
        f"Existen valores numéricos inválidos: {campo}"
    )

# Variable objetivo
assert (
    datos_aptos["no_asistio"]
    .astype("string")
    .str.strip()
    .isin(["0", "1"])
    .all()
), "Existen resultados de asistencia inválidos."

# Costo
assert pd.to_numeric(
    datos_aptos["costo_estimado_usd"],
    errors="coerce"
).ge(0).all(), "Existen costos inválidos."


# ============================================================
# 16. EVALUAR POSIBLE IMPACTO DE LA SELECCIÓN
# ============================================================

# Distribución de no_asistio antes y después
# para identificar diferencias que requieran análisis.

def distribucion_objetivo(datos, etapa):

    serie = (
        datos["no_asistio"]
        .astype("string")
        .str.strip()
        .fillna("SIN DATO")
        .replace("", "SIN DATO")
    )

    tabla = (
        serie.value_counts()
        .rename_axis("Valor")
        .reset_index(name="Cantidad")
    )

    tabla["Etapa"] = etapa

    tabla["Porcentaje"] = (
        tabla["Cantidad"] / len(datos) * 100
    ).round(2)

    return tabla


impacto_asistencia = pd.concat([
    distribucion_objetivo(
        original,
        "ORIGINAL"
    ),
    distribucion_objetivo(
        datos_aptos,
        "CONJUNTO_APTO"
    )
], ignore_index=True)


# ============================================================
# 17. MOSTRAR RESULTADOS
# ============================================================

print("\n" + "=" * 70)
print("RESULTADOS DE LA LIMPIEZA")
print("=" * 70)

print("\n1. DESTINO DE LOS REGISTROS")
display(resumen_destinos)

print("\n2. COMPARATIVO DE KPI")
display(comparativo_kpis)

print("\n3. COBERTURA DEL CONJUNTO DEPURADO")
print("Originales:", total_original)
print("Aptos:", cantidad_aptos)
print("No aptos:", cantidad_no_aptos)
print(f"Cobertura conservada: {cobertura:.2f}%")

print("\n4. MODIFICACIONES DE FORMATO")
print("Celdas normalizadas:", len(detalle_cambios))

display(detalle_cambios.head(10))

print("\n5. BITÁCORA DE TRATAMIENTOS")
display(bitacora.head(15))

print("\n6. DISTRIBUCIÓN DE INASISTENCIA")
display(impacto_asistencia)

print("\nVERIFICACIONES DEL CONJUNTO APTO: SUPERADAS")


# ============================================================
# 18. GUARDAR EVIDENCIAS
# ============================================================

evidencias = {
    "dataset_apto": datos_aptos,
    "registros_no_aptos": datos_no_aptos,
    "bitacora_tratamientos": bitacora,
    "detalle_normalizaciones": detalle_cambios,
    "comparativo_kpi_antes_despues": comparativo_kpis,
    "resumen_destinos": resumen_destinos,
    "impacto_distribucion_asistencia": impacto_asistencia
}

for nombre, tabla in evidencias.items():

    ruta = (
        carpeta_limpieza
        / f"{nombre}_{fecha_limpieza}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)

print("\n" + "=" * 70)
print("PASO 4 FINALIZADO")
print("=" * 70)

print("Carpeta:", carpeta_limpieza)
print("SQL Server no fue modificado.")
print(
    "Los registros en cuarentena requieren revisión "
    "antes de considerarse recuperados."
)
print(
    "El conjunto apto no reemplaza automáticamente "
    "el dataset utilizado por Minería de Datos."
)


PASO 4. LIMPIEZA Y CONTROL DE TRAZABILIDAD

DATOS ORIGINALES
Registros: 10080
Registros completos: 9160
IDs repetidos: 80
Filas duplicadas: 47

NORMALIZACIÓN DE CATEGORÍAS
Valores normalizados: 273

CONTROL DE DUPLICADOS
Copias idénticas adicionales tras normalización: 80
Registros pendientes por conflictos de ID: 0

KPI OFICIALES DEL II CORTE


,Codigo,KPI,Antes,Despues,Meta,Cumple_despues
0,IND-01,Tasa de integridad,90.87,100.0,>98%,SI
1,IND-02,Porcentaje de categorías no normalizadas,2.71,0.0,<1%,SI
2,IND-04,Porcentaje de filas duplicadas,0.47,0.0,0%,SI



CONTROL DE CATEGORÍAS
No normalizados antes: 273
No normalizados después: 0

RESULTADOS DE LA LIMPIEZA

1. DESTINO DE LOS REGISTROS


,Estado,Registros,Porcentaje
0,APTO,9076,90.04
1,EN_CUARENTENA_PARA_REVISION,924,9.17
2,EXCLUIDO_COPIA_CONFIRMADA,80,0.79



2. COMPARATIVO DE KPI


...


5. BITÁCORA DE TRATAMIENTOS


,id_raw,id_cita,Fuente,Motivos,Campos_faltantes,Cantidad_motivos,Estado_final,Fecha_proceso_UTC,Observacion
0,1,RSV2-000001,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server
1,2,RSV2-000002,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server
2,3,RSV2-000003,dbo.raw_cita,POL02_REGISTRO_INCOMPLETO,grupo_edad,1,EN_CUARENTENA_PARA_REVISION,20261024T210524Z,Registro original conservado en SQL Server
3,4,RSV2-000004,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server
4,5,RSV2-000005,dbo.raw_cita,POL02_REGISTRO_INCOMPLETO,no_asistio,1,EN_CUARENTENA_PARA_REVISION,20261024T210524Z,Registro original conservado en SQL Server
5,6,RSV2-000006,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server
6,7,RSV2-000007,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server
7,8,RSV2-000008,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server
8,9,RSV2-000009,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server
9,10,RSV2-000010,dbo.raw_cita,,,0,APTO,20261024T210524Z,Registro original conservado en SQL Server



6. DISTRIBUCIÓN DE INASISTENCIA


,Valor,Cantidad,Etapa,Porcentaje
0,0,6263,ORIGINAL,62.13
1,1,3737,ORIGINAL,37.07
2,SIN DATO,80,ORIGINAL,0.79
3,0,5669,CONJUNTO_APTO,62.46
4,1,3407,CONJUNTO_APTO,37.54



VERIFICACIONES DEL CONJUNTO APTO: SUPERADAS
Guardado: dataset_apto_20261024T210524Z.csv
Guardado: registros_no_aptos_20261024T210524Z.csv
Guardado: bitacora_tratamientos_20261024T210524Z.csv
Guardado: detalle_normalizaciones_20261024T210524Z.csv
Guardado: comparativo_kpi_antes_despues_20261024T210524Z.csv
Guardado: resumen_destinos_20261024T210524Z.csv
Guardado: impacto_distribucion_asistencia_20261024T210524Z.csv

PASO 4 FINALIZADO
Carpeta: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_III\limpieza
SQL Server no fue modificado.
Los registros en cuarentena requieren revisión antes de considerarse recuperados.
El conjunto apto no reemplaza automáticamente el dataset utilizado por Minería de Datos.



## PASO 5. Ciclo PHVA aplicado al proceso de calidad

### Objetivo
Documentar la planificación, ejecución, verificación y seguimiento del proceso de depuración.

### Evidencias
- Indicadores de calidad antes y después.
- Registros aptos, excluidos y en cuarentena.
- Normalizaciones realizadas.
- Acciones pendientes de revisión.
- Registro del ciclo PHVA.

La mejora de los KPI corresponde al conjunto apto. Los registros originales se conservan sin modificaciones.


In [5]:

# ============================================================
# OPTATIVO III - GESTIÓN DE LA CALIDAD DE LOS PROCESOS
# PASO 5. REGISTRO DEL CICLO PHVA
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

# ============================================================
# 1. VERIFICAR LOS RESULTADOS DEL PASO 4
# ============================================================

variables_necesarias = [
    "original",
    "datos_aptos",
    "bitacora",
    "comparativo_kpis",
    "detalle_cambios"
]

faltantes = [
    nombre
    for nombre in variables_necesarias
    if nombre not in globals()
]

if faltantes:
    raise RuntimeError(
        "Ejecuta el PASO 4. Faltan: "
        + ", ".join(faltantes)
    )

carpeta_phva = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "PHVA"
)

carpeta_phva.mkdir(
    parents=True,
    exist_ok=True
)

fecha = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

# ============================================================
# 2. RESUMIR LOS TRATAMIENTOS
# ============================================================

total_original = len(original)
total_apto = len(datos_aptos)

estados = bitacora["Estado_final"].value_counts()

cuarentena = int(
    estados.get("EN_CUARENTENA_PARA_REVISION", 0)
)

copias_excluidas = int(
    estados.get("EXCLUIDO_COPIA_CONFIRMADA", 0)
)

normalizaciones = len(detalle_cambios)

cobertura = (
    total_apto / total_original * 100
)

assert (
    total_apto + cuarentena + copias_excluidas
    == total_original
), "El balance de registros no coincide."

# ============================================================
# 3. IDENTIFICAR LAS DECISIONES POR POLÍTICA
# ============================================================

motivos = bitacora["Motivos"].fillna("")

seguimiento = pd.DataFrame([
    {
        "Politica": "POL-02",
        "Problema": "Campos obligatorios incompletos",
        "Registros_identificados": int(
            motivos.str.contains(
                "POL02_REGISTRO_INCOMPLETO",
                regex=False
            ).sum()
        ),
        "Tratamiento": "Separación para revisión",
        "Accion_pendiente": (
            "Recuperar datos desde fuentes confiables "
            "o justificar la exclusión"
        )
    },
    {
        "Politica": "POL-03",
        "Problema": "Diferencias de formato",
        "Registros_identificados": int(
            detalle_cambios["id_raw"].nunique()
        ),
        "Tratamiento": (
            "Normalización en copia de trabajo"
        ),
        "Accion_pendiente": (
            "Validar los catálogos provisionales "
            "con el responsable"
        )
    },
    {
        "Politica": "POL-04",
        "Problema": "Duplicidad de registros",
        "Registros_identificados": int(
            motivos.str.contains(
                "POL04_COPIA_DUPLICADA",
                regex=False
            ).sum()
        ),
        "Tratamiento": (
            "Exclusión de copias idénticas adicionales"
        ),
        "Accion_pendiente": (
            "Verificar criterios de deduplicación "
            "y documentar su aprobación"
        )
    },
    {
        "Politica": "POL-05",
        "Problema": "Valores anómalos",
        "Registros_identificados": int(
            bitacora.loc[
                motivos.str.contains(
                    "POL05_",
                    regex=False
                ),
                "id_raw"
            ].nunique()
        ),
        "Tratamiento": (
            "Separación de registros con anomalías"
        ),
        "Accion_pendiente": (
            "Consultar los valores correctos "
            "en la fuente original"
        )
    }
])

# ============================================================
# 4. CONSTRUIR EL REGISTRO PHVA
# ============================================================

registro_phva = pd.DataFrame([
    {
        "Etapa": "PLANIFICAR",
        "Actividad": (
            "Identificar problemas de completitud, "
            "duplicidad, estandarización y validez"
        ),
        "Evidencia": (
            "EDA, auditoría integral y tres KPI iniciales"
        ),
        "Resultado": (
            f"{total_original} registros evaluados"
        ),
        "Estado": "EJECUTADO"
    },
    {
        "Etapa": "HACER",
        "Actividad": (
            "Normalizar formatos, separar registros "
            "no verificables y retirar copias adicionales"
        ),
        "Evidencia": (
            "Bitácora de tratamientos y detalle "
            "de normalizaciones"
        ),
        "Resultado": (
            f"{normalizaciones} normalizaciones; "
            f"{copias_excluidas} copias excluidas; "
            f"{cuarentena} registros en cuarentena"
        ),
        "Estado": "EJECUTADO EN COPIA"
    },
    {
        "Etapa": "VERIFICAR",
        "Actividad": (
            "Recalcular los tres KPI y comprobar "
            "la cobertura de registros"
        ),
        "Evidencia": (
            "Comparativo de KPI antes y después"
        ),
        "Resultado": (
            f"{total_apto} registros aptos; "
            f"cobertura {cobertura:.2f}%"
        ),
        "Estado": "EJECUTADO"
    },
    {
        "Etapa": "ACTUAR",
        "Actividad": (
            "Dar seguimiento a registros en cuarentena "
            "y formalizar controles para nuevas cargas"
        ),
        "Evidencia": (
            "Plan de acciones pendientes por política"
        ),
        "Resultado": (
            "Seguimiento y aprobación institucional pendientes"
        ),
        "Estado": "PENDIENTE DE IMPLEMENTACIÓN"
    }
])

# ============================================================
# 5. VERIFICAR KPI POSTERIORES
# ============================================================

kpis_cumplidos = (
    comparativo_kpis["Cumple_despues"]
    .astype("string")
    .str.upper()
    .eq("SI")
    .sum()
)

print("=" * 70)
print("CICLO PHVA - RED SALUD INTEGRAL CARAZO")
print("=" * 70)

print("\n1. REGISTRO DEL CICLO PHVA")
display(registro_phva)

print("\n2. KPI ANTES Y DESPUÉS")
display(comparativo_kpis)

print("\n3. SEGUIMIENTO POR POLÍTICA")
display(seguimiento)

print("\n4. RESUMEN")
print("Registros originales:", total_original)
print("Registros aptos:", total_apto)
print("Registros en cuarentena:", cuarentena)
print("Copias excluidas:", copias_excluidas)
print("Normalizaciones realizadas:", normalizaciones)
print(f"Cobertura conservada: {cobertura:.2f}%")
print(
    f"KPI cumplidos en el conjunto apto: "
    f"{kpis_cumplidos}/3"
)

# ============================================================
# 6. GUARDAR EVIDENCIAS
# ============================================================

archivos = {
    "registro_ciclo_phva": registro_phva,
    "seguimiento_politicas_phva": seguimiento,
    "comparativo_kpi_phva": comparativo_kpis
}

for nombre, tabla in archivos.items():

    ruta = (
        carpeta_phva
        / f"{nombre}_{fecha}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)

print("\nPASO 5 FINALIZADO")
print("Carpeta:", carpeta_phva)
print("No se modificó SQL Server.")


CICLO PHVA - RED SALUD INTEGRAL CARAZO

1. REGISTRO DEL CICLO PHVA


,Etapa,Actividad,Evidencia,Resultado,Estado
0,PLANIFICAR,"Identificar problemas de completitud, duplicid...","EDA, auditoría integral y tres KPI iniciales",10080 registros evaluados,EJECUTADO
1,HACER,"Normalizar formatos, separar registros no veri...",Bitácora de tratamientos y detalle de normaliz...,273 normalizaciones; 80 copias excluidas; 924 ...,EJECUTADO EN COPIA
2,VERIFICAR,Recalcular los tres KPI y comprobar la cobertu...,Comparativo de KPI antes y después,9076 registros aptos; cobertura 90.04%,EJECUTADO
3,ACTUAR,Dar seguimiento a registros en cuarentena y fo...,Plan de acciones pendientes por política,Seguimiento y aprobación institucional pendientes,PENDIENTE DE IMPLEMENTACIÓN



2. KPI ANTES Y DESPUÉS


,Codigo,KPI,Antes,Despues,Meta,Cumple_despues
0,IND-01,Tasa de integridad,90.87,100.0,>98%,SI
1,IND-02,Porcentaje de categorías no normalizadas,2.71,0.0,<1%,SI
2,IND-04,Porcentaje de filas duplicadas,0.47,0.0,0%,SI



3. SEGUIMIENTO POR POLÍTICA


,Politica,Problema,Registros_identificados,Tratamiento,Accion_pendiente
0,POL-02,Campos obligatorios incompletos,920,Separación para revisión,Recuperar datos desde fuentes confiables o jus...
1,POL-03,Diferencias de formato,273,Normalización en copia de trabajo,Validar los catálogos provisionales con el res...
2,POL-04,Duplicidad de registros,80,Exclusión de copias idénticas adicionales,Verificar criterios de deduplicación y documen...
3,POL-05,Valores anómalos,3,Separación de registros con anomalías,Consultar los valores correctos en la fuente o...



4. RESUMEN
Registros originales: 10080
Registros aptos: 9076
Registros en cuarentena: 924
Copias excluidas: 80
Normalizaciones realizadas: 273
Cobertura conservada: 90.04%
KPI cumplidos en el conjunto apto: 3/3
Guardado: registro_ciclo_phva_20261024T210912Z.csv
Guardado: seguimiento_politicas_phva_20261024T210912Z.csv
Guardado: comparativo_kpi_phva_20261024T210912Z.csv

PASO 5 FINALIZADO
Carpeta: c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_III\PHVA
No se modificó SQL Server.


In [6]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO III - GESTIÓN DE LA CALIDAD
# PASO 6. CONTROL PREVENTIVO Y PRUEBAS
# ============================================================

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

# ============================================================
# 1. PREPARAR REFERENCIAS
# ============================================================

# Se utilizan los resultados de los pasos anteriores.
base = datos_aptos.iloc[0][columnas_originales].to_dict()

ids_existentes = set(
    original["id_cita"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.upper()
)

catalogos_norm = {
    campo: {
        normalizar_categoria(valor)
        for valor in valores
    }
    for campo, valores in catalogos_referencia.items()
}

fecha_prueba = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")


# ============================================================
# 2. FUNCIÓN DE CONTROL DE CALIDAD
# ============================================================

def validar_lote(nuevos_registros):

    datos = nuevos_registros.reset_index(
        drop=True
    ).copy()

    faltan = (
        set(columnas_originales) - set(datos.columns)
    )

    if faltan:
        raise ValueError(
            f"Faltan columnas: {sorted(faltan)}"
        )

    def vacio(valor):
        if pd.isna(valor):
            return True
        return isinstance(valor, str) and not valor.strip()

    ids = (
        datos["id_cita"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    duplicados_lote = ids.duplicated(
        keep=False
    )

    detalles = []

    for indice, fila in datos.iterrows():

        errores = []

        # POL-02: Completitud
        for campo in campos_obligatorios:
            if vacio(fila[campo]):
                errores.append(
                    f"POL-02: {campo} vacío"
                )

        # POL-03: Estandarización
        for campo, permitidos in catalogos_norm.items():

            valor = fila[campo]

            if vacio(valor):
                continue

            clave = normalizar_categoria(valor)

            if clave not in permitidos:
                errores.append(
                    f"POL-03: {campo} fuera de catálogo"
                )

            elif str(valor) not in catalogos_referencia[campo]:
                errores.append(
                    f"POL-03: {campo} requiere normalización"
                )

        # POL-04: Duplicidad
        if duplicados_lote.iloc[indice]:
            errores.append(
                "POL-04: ID repetido dentro del lote"
            )

        if ids.iloc[indice] in ids_existentes:
            errores.append(
                "POL-04: ID existente en el origen"
            )

        # POL-05: Valores numéricos
        for campo in campos_numericos:

            valor = fila[campo]

            if vacio(valor):
                continue

            numero = pd.to_numeric(
                pd.Series([valor]),
                errors="coerce"
            ).iloc[0]

            if (
                pd.isna(numero)
                or numero < 0
                or numero % 1 != 0
            ):
                errores.append(
                    f"POL-05: {campo} inválido"
                )

        # POL-05: Resultado de asistencia
        if not vacio(fila["no_asistio"]):

            if str(fila["no_asistio"]).strip() not in ["0", "1"]:
                errores.append(
                    "POL-05: no_asistio inválido"
                )

        # Revisión adicional de costo
        costo = pd.to_numeric(
            pd.Series([fila["costo_estimado_usd"]]),
            errors="coerce"
        ).iloc[0]

        if pd.isna(costo) or costo < 0:
            errores.append(
                "EDA: costo_estimado_usd requiere revisión"
            )

        detalles.append({
            "id_cita": fila["id_cita"],
            "Estado": (
                "RECHAZADO" if errores else "ACEPTADO"
            ),
            "Cantidad_errores": len(errores),
            "Detalle": " | ".join(errores)
        })

    return pd.DataFrame(detalles)


# ============================================================
# 3. CREAR REGISTROS DE PRUEBA
# ============================================================

def crear_registro(sufijo, cambios=None):

    registro = base.copy()

    registro["id_cita"] = (
        f"PRUEBA-CALIDAD-{fecha_prueba}-{sufijo}"
    )

    if cambios:
        registro.update(cambios)

    return registro


casos = [
    (
        "POL-02",
        "Registro completo",
        "ACEPTADO",
        [crear_registro("01")]
    ),
    (
        "POL-02",
        "Centro de salud vacío",
        "RECHAZADO",
        [crear_registro("02", {
            "centro_salud": None
        })]
    ),
    (
        "POL-03",
        "Categoría válida",
        "ACEPTADO",
        [crear_registro("03")]
    ),
    (
        "POL-03",
        "Categoría desconocida",
        "RECHAZADO",
        [crear_registro("04", {
            "centro_salud": "CENTRO DESCONOCIDO"
        })]
    ),
    (
        "POL-04",
        "Identificadores diferentes",
        "ACEPTADO",
        [
            crear_registro("05A"),
            crear_registro("05B")
        ]
    ),
    (
        "POL-04",
        "Identificador repetido",
        "RECHAZADO",
        [
            crear_registro("06"),
            crear_registro("06")
        ]
    ),
    (
        "POL-05",
        "Días de espera válidos",
        "ACEPTADO",
        [crear_registro("07", {
            "espera_dias": 5
        })]
    ),
    (
        "POL-05",
        "Días de espera negativos",
        "RECHAZADO",
        [crear_registro("08", {
            "espera_dias": -5
        })]
    )
]


# ============================================================
# 4. EJECUTAR LAS PRUEBAS
# ============================================================

resultados = []
detalle_pruebas = []

for politica, caso, esperado, registros in casos:

    salida = validar_lote(
        pd.DataFrame(registros)
    )

    obtenido = (
        "ACEPTADO"
        if salida["Estado"].eq("ACEPTADO").all()
        else "RECHAZADO"
    )

    resultado = (
        "SUPERADA"
        if obtenido == esperado
        else "FALLIDA"
    )

    resultados.append({
        "Politica": politica,
        "Caso": caso,
        "Esperado": esperado,
        "Obtenido": obtenido,
        "Resultado": resultado
    })

    detalle = salida.copy()
    detalle["Caso"] = caso
    detalle["Politica_prueba"] = politica

    detalle_pruebas.append(detalle)

tabla_pruebas = pd.DataFrame(resultados)

detalle_pruebas = pd.concat(
    detalle_pruebas,
    ignore_index=True
)


# ============================================================
# 5. MOSTRAR RESULTADOS
# ============================================================

print("=" * 70)
print("PASO 6. CONTROL PREVENTIVO DE CALIDAD")
print("=" * 70)

print("\nPRUEBAS FUNCIONALES")
display(tabla_pruebas)

print("\nDETALLE DE REGISTROS PROBADOS")
display(detalle_pruebas)

superadas = int(
    tabla_pruebas["Resultado"].eq("SUPERADA").sum()
)

print(
    f"\nPruebas superadas: "
    f"{superadas}/{len(tabla_pruebas)}"
)


# ============================================================
# 6. GUARDAR EVIDENCIAS
# ============================================================

carpeta_controles = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "PHVA"
    / "controles_preventivos"
)

carpeta_controles.mkdir(
    parents=True,
    exist_ok=True
)

tabla_pruebas.to_csv(
    carpeta_controles
    / f"pruebas_control_{fecha_prueba}.csv",
    index=False,
    encoding="utf-8-sig"
)

detalle_pruebas.to_csv(
    carpeta_controles
    / f"detalle_control_{fecha_prueba}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nEvidencias guardadas en:")
print(carpeta_controles)

print("\nNo se insertaron registros de prueba en SQL Server.")

if not tabla_pruebas["Resultado"].eq("SUPERADA").all():
    raise AssertionError(
        "Existen pruebas fallidas. Revisar el control."
    )

print("\nCONTROL PREVENTIVO: PRUEBAS SUPERADAS")


PASO 6. CONTROL PREVENTIVO DE CALIDAD

PRUEBAS FUNCIONALES


,Politica,Caso,Esperado,Obtenido,Resultado
0,POL-02,Registro completo,ACEPTADO,ACEPTADO,SUPERADA
1,POL-02,Centro de salud vacío,RECHAZADO,RECHAZADO,SUPERADA
2,POL-03,Categoría válida,ACEPTADO,ACEPTADO,SUPERADA
3,POL-03,Categoría desconocida,RECHAZADO,RECHAZADO,SUPERADA
4,POL-04,Identificadores diferentes,ACEPTADO,ACEPTADO,SUPERADA
5,POL-04,Identificador repetido,RECHAZADO,RECHAZADO,SUPERADA
6,POL-05,Días de espera válidos,ACEPTADO,ACEPTADO,SUPERADA
7,POL-05,Días de espera negativos,RECHAZADO,RECHAZADO,SUPERADA



DETALLE DE REGISTROS PROBADOS


,id_cita,Estado,Cantidad_errores,Detalle,Caso,Politica_prueba
0,PRUEBA-CALIDAD-20261024T184715Z-01,ACEPTADO,0,,Registro completo,POL-02
1,PRUEBA-CALIDAD-20261024T184715Z-02,RECHAZADO,1,POL-02: centro_salud vacío,Centro de salud vacío,POL-02
2,PRUEBA-CALIDAD-20261024T184715Z-03,ACEPTADO,0,,Categoría válida,POL-03
3,PRUEBA-CALIDAD-20261024T184715Z-04,RECHAZADO,1,POL-03: centro_salud fuera de catálogo,Categoría desconocida,POL-03
4,PRUEBA-CALIDAD-20261024T184715Z-05A,ACEPTADO,0,,Identificadores diferentes,POL-04
5,PRUEBA-CALIDAD-20261024T184715Z-05B,ACEPTADO,0,,Identificadores diferentes,POL-04
6,PRUEBA-CALIDAD-20261024T184715Z-06,RECHAZADO,1,POL-04: ID repetido dentro del lote,Identificador repetido,POL-04
7,PRUEBA-CALIDAD-20261024T184715Z-06,RECHAZADO,1,POL-04: ID repetido dentro del lote,Identificador repetido,POL-04
8,PRUEBA-CALIDAD-20261024T184715Z-07,ACEPTADO,0,,Días de espera válidos,POL-05
9,PRUEBA-CALIDAD-20261024T184715Z-08,RECHAZADO,1,POL-05: espera_dias inválido,Días de espera negativos,POL-05



Pruebas superadas: 8/8

Evidencias guardadas en:
c:\Users\Usuario\Downloads\RedSaludCarazo_proyecto_base\resultados_jupyter\optativo_III\PHVA\controles_preventivos

No se insertaron registros de prueba en SQL Server.

CONTROL PREVENTIVO: PRUEBAS SUPERADAS



## PASO 7. Consolidación final de evidencias

### Objetivo
Consolidar los resultados del análisis exploratorio, la auditoría integral, la depuración, los tres KPI de calidad, el ciclo PHVA y las pruebas del validador.

### Alcance
El proceso utiliza los registros originales de Red Salud Integral Carazo y un conjunto depurado independiente.

### Evidencias
- Balance de registros originales y depurados.
- Indicadores de calidad antes y después.
- Auditoría y tratamientos por identificador.
- Comparación de la distribución de los datos.
- Registro del ciclo PHVA.
- Resultados de pruebas funcionales.

Las metas de los KPI se verifican sobre el conjunto apto, manteniendo visible la cobertura respecto a los registros originales.


In [7]:

# ============================================================
# RED SALUD INTEGRAL CARAZO
# OPTATIVO III - GESTIÓN DE LA CALIDAD DE LOS PROCESOS
# PASO 7. CONSOLIDACIÓN FINAL DE EVIDENCIAS
# ============================================================

import pandas as pd

from datetime import datetime, timezone
from IPython.display import display


# ============================================================
# 1. VERIFICAR QUE LOS PASOS ANTERIORES SE EJECUTARON
# ============================================================

variables_necesarias = [
    "original",
    "datos_aptos",
    "bitacora",
    "auditoria_general",
    "detalle_cambios",
    "comparativo_kpis",
    "registro_phva",
    "seguimiento",
    "tabla_pruebas",
    "detalle_pruebas"
]

variables_faltantes = [
    nombre
    for nombre in variables_necesarias
    if nombre not in globals()
]

if variables_faltantes:
    raise RuntimeError(
        "Debes ejecutar los pasos anteriores. "
        "Faltan estas variables: "
        + ", ".join(variables_faltantes)
    )


# ============================================================
# 2. CREAR CARPETA DEL RESUMEN FINAL
# ============================================================

carpeta_final = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "resumen_final"
)

carpeta_final.mkdir(
    parents=True,
    exist_ok=True
)

fecha_final = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")


# ============================================================
# 3. BALANCE Y TRAZABILIDAD DE REGISTROS
# ============================================================

total_original = len(original)

total_aptos = len(datos_aptos)

estados = bitacora["Estado_final"].value_counts()

total_cuarentena = int(
    estados.get(
        "EN_CUARENTENA_PARA_REVISION",
        0
    )
)

total_excluidos = int(
    estados.get(
        "EXCLUIDO_COPIA_CONFIRMADA",
        0
    )
)

cobertura = (
    total_aptos / total_original * 100
)

# Verificar que ningún registro original desaparezca
# del control de trazabilidad.

assert (
    total_original
    == total_aptos + total_cuarentena + total_excluidos
), "Error en el balance de registros."

assert (
    bitacora["id_raw"].nunique() == total_original
), "Faltan registros en la bitácora."

assert (
    original["id_raw"].is_unique
), "La clave id_raw del origen no es única."

assert (
    set(original["id_raw"])
    == set(bitacora["id_raw"])
), "Los identificadores de la bitácora no coinciden."

assert (
    set(datos_aptos["id_raw"])
    == set(
        bitacora.loc[
            bitacora["Estado_final"] == "APTO",
            "id_raw"
        ]
    )
), "El conjunto apto no coincide con la bitácora."

balance_final = pd.DataFrame([
    {
        "Concepto": "Registros originales",
        "Cantidad": total_original,
        "Porcentaje": 100.0
    },
    {
        "Concepto": "Registros aptos",
        "Cantidad": total_aptos,
        "Porcentaje": round(cobertura, 2)
    },
    {
        "Concepto": "Registros en cuarentena",
        "Cantidad": total_cuarentena,
        "Porcentaje": round(
            total_cuarentena / total_original * 100, 2
        )
    },
    {
        "Concepto": "Copias adicionales excluidas",
        "Cantidad": total_excluidos,
        "Porcentaje": round(
            total_excluidos / total_original * 100, 2
        )
    }
])


# ============================================================
# 4. CONSOLIDAR LOS KPI DEL II CORTE
# ============================================================

kpis_finales = comparativo_kpis.copy()

assert (
    set(kpis_finales["Codigo"])
    == {"IND-01", "IND-04", "IND-05"}
), "Los KPI no corresponden a los tres seleccionados."

cantidad_kpi_cumplidos = int(
    kpis_finales["Cumple_despues"]
    .astype("string")
    .str.upper()
    .eq("SI")
    .sum()
)


# ============================================================
# 5. RESUMEN DE HALLAZGOS DE AUDITORÍA
# ============================================================

auditoria_resumen_final = (
    auditoria_general
    .groupby("Politica")
    .agg(
        Incidencias=("id_raw", "size"),
        Registros_afectados=("id_raw", "nunique")
    )
    .reset_index()
)

# Un registro puede presentar varios problemas.
total_registros_con_hallazgos = int(
    auditoria_general["id_raw"].nunique()
)


# ============================================================
# 6. VERIFICAR NORMALIZACIONES
# ============================================================

total_normalizaciones = len(detalle_cambios)

registros_normalizados = int(
    detalle_cambios["id_raw"].nunique()
)

normalizados_aptos = int(
    detalle_cambios.loc[
        detalle_cambios["id_raw"].isin(
            datos_aptos["id_raw"]
        ),
        "id_raw"
    ].nunique()
)

resumen_normalizaciones = pd.DataFrame([
    {
        "Concepto": "Celdas normalizadas",
        "Cantidad": total_normalizaciones
    },
    {
        "Concepto": "Registros con normalizaciones",
        "Cantidad": registros_normalizados
    },
    {
        "Concepto": "Registros normalizados que quedaron aptos",
        "Cantidad": normalizados_aptos
    }
])


# ============================================================
# 7. VERIFICAR LAS PRUEBAS FUNCIONALES
# ============================================================

pruebas_finales = tabla_pruebas.copy()

pruebas_superadas = int(
    pruebas_finales["Resultado"]
    .eq("SUPERADA")
    .sum()
)

total_pruebas = len(pruebas_finales)

assert (
    total_pruebas == 8
), "No se encontraron las ocho pruebas previstas."

assert (
    pruebas_superadas == total_pruebas
), "Existen pruebas funcionales no superadas."


# ============================================================
# 8. COMPARAR LA DISTRIBUCIÓN DE DATOS
# ============================================================

def crear_distribucion(datos, campo, etapa):

    serie = (
        datos[campo]
        .astype("string")
        .str.strip()
        .fillna("SIN DATO")
        .replace("", "SIN DATO")
    )

    tabla = (
        serie.value_counts()
        .rename_axis("Valor")
        .reset_index(name="Cantidad")
    )

    tabla["Variable"] = campo
    tabla["Etapa"] = etapa

    tabla["Porcentaje"] = (
        tabla["Cantidad"] / len(datos) * 100
    ).round(2)

    return tabla


variables_comparacion = [
    "no_asistio",
    "centro_salud",
    "especialidad",
    "canal_agendamiento",
    "grupo_edad"
]

distribuciones = []

for campo in variables_comparacion:

    distribuciones.append(
        crear_distribucion(
            original,
            campo,
            "ORIGINAL"
        )
    )

    distribuciones.append(
        crear_distribucion(
            datos_aptos,
            campo,
            "CONJUNTO_APTO"
        )
    )

comparacion_distribuciones = pd.concat(
    distribuciones,
    ignore_index=True
)

# Comparar los resultados de asistencia.
# No se utilizan estos porcentajes para asegurar
# que no existe sesgo; son evidencia descriptiva.

distribucion_asistencia = (
    comparacion_distribuciones.loc[
        comparacion_distribuciones["Variable"]
        == "no_asistio"
    ]
    .copy()
)


# ============================================================
# 9. COMPROBAR EL REGISTRO PHVA
# ============================================================

etapas_esperadas = {
    "PLANIFICAR",
    "HACER",
    "VERIFICAR",
    "ACTUAR"
}

assert (
    set(registro_phva["Etapa"])
    == etapas_esperadas
), "El registro PHVA no contiene las cuatro etapas."


# ============================================================
# 10. MOSTRAR EL RESUMEN FINAL
# ============================================================

print("=" * 70)
print("RED SALUD INTEGRAL CARAZO")
print("OPTATIVO III - RESUMEN FINAL DE CALIDAD")
print("=" * 70)

print("\n1. BALANCE DE REGISTROS")
display(balance_final)

print("\n2. KPI ANTES Y DESPUÉS")
display(kpis_finales)

print("\n3. AUDITORÍA POR POLÍTICA")
display(auditoria_resumen_final)

print(
    "\nRegistros únicos con al menos un hallazgo:",
    total_registros_con_hallazgos
)

print("\n4. NORMALIZACIONES")
display(resumen_normalizaciones)

print("\n5. CICLO PHVA")
display(registro_phva)

print("\n6. PRUEBAS FUNCIONALES")
display(pruebas_finales)

print("\n7. DISTRIBUCIÓN DE INASISTENCIA")
display(distribucion_asistencia)

print("\n" + "=" * 70)
print("VERIFICACIÓN GENERAL")
print("=" * 70)

print("Registros originales:", total_original)
print("Registros aptos:", total_aptos)
print("Registros en cuarentena:", total_cuarentena)
print("Copias adicionales excluidas:", total_excluidos)

print(f"Cobertura conservada: {cobertura:.2f}%")

print(
    f"KPI cumplidos en el conjunto apto: "
    f"{cantidad_kpi_cumplidos}/3"
)

print(
    f"Pruebas funcionales superadas: "
    f"{pruebas_superadas}/{total_pruebas}"
)


# ============================================================
# 11. GUARDAR TODAS LAS EVIDENCIAS
# ============================================================

evidencias_finales = {
    "balance_registros": balance_final,
    "kpi_antes_despues": kpis_finales,
    "resumen_auditoria": auditoria_resumen_final,
    "trazabilidad_registros": bitacora,
    "normalizaciones": detalle_cambios,
    "resumen_normalizaciones": resumen_normalizaciones,
    "registro_phva": registro_phva,
    "seguimiento_phva": seguimiento,
    "pruebas_funcionales": pruebas_finales,
    "detalle_pruebas": detalle_pruebas,
    "distribuciones_antes_despues":
        comparacion_distribuciones
}

for nombre, tabla in evidencias_finales.items():

    ruta = (
        carpeta_final
        / f"{nombre}_{fecha_final}.csv"
    )

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", ruta.name)


# ============================================================
# 12. OBSERVACIONES PARA LA PRESENTACIÓN
# ============================================================

print("\n" + "=" * 70)
print("OBSERVACIONES FINALES")
print("=" * 70)

print(
    "- Los KPI posteriores corresponden "
    "al conjunto apto, no a todos los registros originales."
)

print(
    "- Los registros en cuarentena todavía "
    "requieren revisión de sus valores."
)

print(
    "- Las categorías se normalizaron usando "
    "catálogos provisionales."
)

print(
    "- La integración del control preventivo "
    "en un sistema de captura sigue pendiente."
)

print(
    "- La base de datos original no fue modificada."
)

print("\nPASO 7 FINALIZADO")
print("Evidencias guardadas en:")
print(carpeta_final)


RED SALUD INTEGRAL CARAZO
OPTATIVO III - RESUMEN FINAL DE CALIDAD

1. BALANCE DE REGISTROS


,Concepto,Cantidad,Porcentaje
0,Registros originales,10080,100.00
1,Registros aptos,9076,90.04
2,Registros en cuarentena,924,9.17
3,Copias adicionales excluidas,80,0.79



2. KPI ANTES Y DESPUÉS


,Codigo,KPI,Antes,Despues,Meta,Cumple_despues
0,IND-01,Tasa de integridad,90.87,100.0,>98%,SI
1,IND-04,Porcentaje de filas duplicadas,0.47,0.0,0%,SI
2,IND-05,Tasa de duplicidad de IDs,0.79,0.0,0%,SI



3. AUDITORÍA POR POLÍTICA


...


6. PRUEBAS FUNCIONALES


,Politica,Caso,Esperado,Obtenido,Resultado
0,POL-02,Registro completo,ACEPTADO,ACEPTADO,SUPERADA
1,POL-02,Centro de salud vacío,RECHAZADO,RECHAZADO,SUPERADA
2,POL-03,Categoría válida,ACEPTADO,ACEPTADO,SUPERADA
3,POL-03,Categoría desconocida,RECHAZADO,RECHAZADO,SUPERADA
4,POL-04,Identificadores diferentes,ACEPTADO,ACEPTADO,SUPERADA
5,POL-04,Identificador repetido,RECHAZADO,RECHAZADO,SUPERADA
6,POL-05,Días de espera válidos,ACEPTADO,ACEPTADO,SUPERADA
7,POL-05,Días de espera negativos,RECHAZADO,RECHAZADO,SUPERADA



7. DISTRIBUCIÓN DE INASISTENCIA


,Valor,Cantidad,Variable,Etapa,Porcentaje
0,0,6263,no_asistio,ORIGINAL,62.13
1,1,3737,no_asistio,ORIGINAL,37.07
2,SIN DATO,80,no_asistio,ORIGINAL,0.79
3,0,5669,no_asistio,CONJUNTO_APTO,62.46
4,1,3407,no_asistio,CONJUNTO_APTO,37.54



VERIFICACIÓN GENERAL
Registros originales: 10080
Registros aptos: 9076
Registros en cuarentena: 924
Copias adicionales excluidas: 80
Cobertura conservada: 90.04%
KPI cumplidos en el conjunto apto: 3/3
Pruebas funcionales superadas: 8/8
Guardado: balance_registros_20261024T185136Z.csv
Guardado: kpi_antes_despues_20261024T185136Z.csv
Guardado: resumen_auditoria_20261024T185136Z.csv
Guardado: trazabilidad_registros_20261024T185136Z.csv
Guardado: normalizaciones_20261024T185136Z.csv
Guardado: resumen_normalizaciones_20261024T185136Z.csv
Guardado: registro_phva_20261024T185136Z.csv
Guardado: seguimiento_phva_20261024T185136Z.csv
Guardado: pruebas_funcionales_20261024T185136Z.csv
Guardado: detalle_pruebas_20261024T185136Z.csv
Guardado: distribuciones_antes_despues_20261024T185136Z.csv

OBSERVACIONES FINALES
- Los KPI posteriores corresponden al conjunto apto, no a todos los registros originales.
- Los registros en cuarentena todavía requieren revisión de sus valores.
- Las categorías se nor

In [8]:

# ============================================================
# OPTATIVO III - GESTIÓN DE LA CALIDAD DE LOS PROCESOS
# PASO 4B. KPI OFICIALES DEL II CORTE
#
# IND-01: INTEGRIDAD
# IND-02: CATEGORÍAS NO NORMALIZADAS
# IND-04: FILAS DUPLICADAS
# ============================================================

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

campos_categoricos = [
    "centro_salud",
    "especialidad",
    "canal_agendamiento"
]

fecha_kpi = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

carpeta_kpi = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "resumen_final"
)

carpeta_kpi.mkdir(
    parents=True,
    exist_ok=True
)

carpeta_auditoria_ind02 = (
    proyecto
    / "resultados_jupyter"
    / "optativo_III"
    / "auditoria_integral"
)

carpeta_auditoria_ind02.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FUNCIÓN PARA CALCULAR KPI OFICIALES
# ============================================================

def calcular_kpi_oficiales(datos):

    n = len(datos)

    if n == 0:
        raise ValueError("No hay datos para evaluar.")

    # --------------------------------------------------------
    # IND-01. TASA DE INTEGRIDAD
    # --------------------------------------------------------

    faltantes = pd.DataFrame({
        campo: detectar_faltantes(datos[campo])
        for campo in campos_obligatorios
    })

    registros_completos = int(
        (~faltantes.any(axis=1)).sum()
    )

    integridad = (
        registros_completos / n * 100
    )

    # --------------------------------------------------------
    # IND-02. CATEGORÍAS NO NORMALIZADAS
    # --------------------------------------------------------

    registros_categorizados = pd.Series(
        False,
        index=datos.index
    )

    registros_no_normalizados = pd.Series(
        False,
        index=datos.index
    )

    detalle = []

    for campo in campos_categoricos:

        permitidos = {
            normalizar_categoria(valor): valor
            for valor in catalogos_referencia[campo]
        }

        serie = datos[campo].astype("string")

        tiene_valor = (
            ~detectar_faltantes(serie)
        )

        normalizados = serie.map(
            normalizar_categoria
        )

        valores_esperados = normalizados.map(
            permitidos
        )

        fuera_catalogo = (
            tiene_valor
            & ~normalizados.isin(permitidos)
        )

        formato_diferente = (
            tiene_valor
            & normalizados.isin(permitidos)
            & serie.ne(valores_esperados).fillna(False)
        )

        incumple = (
            fuera_catalogo | formato_diferente
        ).fillna(False)

        registros_categorizados |= tiene_valor
        registros_no_normalizados |= incumple

        for indice in datos.index[incumple]:

            fila = datos.loc[indice]

            detalle.append({
                "id_raw": fila["id_raw"],
                "id_cita": fila["id_cita"],
                "Campo": campo,
                "Valor_original": fila[campo],
                "Valor_esperado": (
                    valores_esperados.loc[indice]
                    if not fuera_catalogo.loc[indice]
                    else "REQUIERE REVISIÓN"
                ),
                "Tipo_hallazgo": (
                    "FUERA_DE_CATALOGO"
                    if fuera_catalogo.loc[indice]
                    else "FORMATO_NO_ESTANDAR"
                ),
                "Estado": "HALLAZGO_DETECTADO"
            })

    total_categorizados = int(
        registros_categorizados.sum()
    )

    total_no_normalizados = int(
        registros_no_normalizados.sum()
    )

    if total_categorizados == 0:
        raise ValueError(
            "No existen registros categorizados."
        )

    porcentaje_no_normalizados = (
        total_no_normalizados
        / total_categorizados
        * 100
    )

    # --------------------------------------------------------
    # IND-04. PORCENTAJE DE FILAS DUPLICADAS
    # --------------------------------------------------------

    filas_duplicadas = int(
        datos.duplicated(
            subset=columnas_originales,
            keep="first"
        ).sum()
    )

    porcentaje_duplicadas = (
        filas_duplicadas / n * 100
    )

    resultados = {
        "Registros": n,
        "Completos": registros_completos,
        "Categorizados": total_categorizados,
        "No_normalizados": total_no_normalizados,
        "Filas_duplicadas": filas_duplicadas,
        "IND-01": integridad,
        "IND-02": porcentaje_no_normalizados,
        "IND-04": porcentaje_duplicadas
    }

    return resultados, pd.DataFrame(
        detalle,
        columns=[
            "id_raw",
            "id_cita",
            "Campo",
            "Valor_original",
            "Valor_esperado",
            "Tipo_hallazgo",
            "Estado"
        ]
    )


# ============================================================
# 3. CALCULAR ANTES Y DESPUÉS
# ============================================================

kpi_originales, detalle_ind02_original = (
    calcular_kpi_oficiales(original)
)

kpi_depurados, detalle_ind02_depurado = (
    calcular_kpi_oficiales(datos_aptos)
)


# ============================================================
# 4. CONSTRUIR COMPARATIVO OFICIAL
# ============================================================

comparativo_kpis = pd.DataFrame([
    {
        "Codigo": "IND-01",
        "KPI": "Tasa de integridad de registros de citas",
        "Antes": round(kpi_originales["IND-01"], 2),
        "Despues": round(kpi_depurados["IND-01"], 2),
        "Meta": ">98%",
        "Cumple_despues": (
            "SI"
            if kpi_depurados["IND-01"] > 98
            else "NO"
        )
    },
    {
        "Codigo": "IND-02",
        "KPI": "Porcentaje de registros con categorías no normalizadas",
        "Antes": round(kpi_originales["IND-02"], 2),
        "Despues": round(kpi_depurados["IND-02"], 2),
        "Meta": "<1%",
        "Cumple_despues": (
            "SI"
            if kpi_depurados["IND-02"] < 1
            else "NO"
        )
    },
    {
        "Codigo": "IND-04",
        "KPI": "Porcentaje de filas duplicadas",
        "Antes": round(kpi_originales["IND-04"], 2),
        "Despues": round(kpi_depurados["IND-04"], 2),
        "Meta": "0%",
        "Cumple_despues": (
            "SI"
            if kpi_depurados["Filas_duplicadas"] == 0
            else "NO"
        )
    }
])


# ============================================================
# 5. MOSTRAR RESULTADOS
# ============================================================

print("=" * 70)
print("KPI OFICIALES DEL II CORTE")
print("OPTATIVO III - GESTIÓN DE LA CALIDAD")
print("=" * 70)

display(comparativo_kpis)

print("\nDETALLE DEL IND-02")

print(
    "Registros categorizados en origen:",
    kpi_originales["Categorizados"]
)

print(
    "Registros no normalizados en origen:",
    kpi_originales["No_normalizados"]
)

print(
    "Registros categorizados en el conjunto apto:",
    kpi_depurados["Categorizados"]
)

print(
    "Registros no normalizados en el conjunto apto:",
    kpi_depurados["No_normalizados"]
)

print("\nEJEMPLOS DE INCUMPLIMIENTOS IND-02")
display(detalle_ind02_original.head(15))

# Comprobar que no existen IDs duplicados en
# el detalle de registros si un mismo registro
# incumple en varios campos; se contabiliza
# una sola vez en el KPI.

assert (
    detalle_ind02_original["id_raw"].nunique()
    == kpi_originales["No_normalizados"]
), "El detalle IND-02 no coincide con el indicador."


# ============================================================
# 6. GUARDAR EVIDENCIAS
# ============================================================

ruta_comparativo = (
    carpeta_kpi
    / f"kpi_oficiales_II_corte_{fecha_kpi}.csv"
)

ruta_detalle = (
    carpeta_auditoria_ind02
    / f"IND02_registros_no_normalizados_{fecha_kpi}.csv"
)

comparativo_kpis.to_csv(
    ruta_comparativo,
    index=False,
    encoding="utf-8-sig"
)

detalle_ind02_original.to_csv(
    ruta_detalle,
    index=False,
    encoding="utf-8-sig"
)

print("\nEVIDENCIAS GUARDADAS")
print("Comparativo:", ruta_comparativo.name)
print("Auditoría IND-02:", ruta_detalle.name)

print("\nPASO 4B FINALIZADO")
print("No se modificaron registros de SQL Server.")


KPI OFICIALES DEL II CORTE
OPTATIVO III - GESTIÓN DE LA CALIDAD


,Codigo,KPI,Antes,Despues,Meta,Cumple_despues
0,IND-01,Tasa de integridad de registros de citas,90.87,100.0,>98%,SI
1,IND-02,Porcentaje de registros con categorías no norm...,2.71,0.0,<1%,SI
2,IND-04,Porcentaje de filas duplicadas,0.47,0.0,0%,SI



DETALLE DEL IND-02
Registros categorizados en origen: 10080
Registros no normalizados en origen: 273
Registros categorizados en el conjunto apto: 9076
Registros no normalizados en el conjunto apto: 0

EJEMPLOS DE INCUMPLIMIENTOS IND-02


,id_raw,id_cita,Campo,Valor_original,Valor_esperado,Tipo_hallazgo,Estado
0,102,RSV2-000102,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
1,258,RSV2-000258,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
2,379,RSV2-000379,centro_salud,dolores,DOLORES,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
3,538,RSV2-000538,centro_salud,dolores,DOLORES,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
4,739,RSV2-000739,centro_salud,dolores,DOLORES,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
5,846,RSV2-000846,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
6,945,RSV2-000945,centro_salud,san marcos,SAN MARCOS,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
7,1174,RSV2-001174,centro_salud,diriamba,DIRIAMBA,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
8,1184,RSV2-001184,centro_salud,jinotepe,JINOTEPE,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO
9,1254,RSV2-001254,centro_salud,diriamba,DIRIAMBA,FORMATO_NO_ESTANDAR,HALLAZGO_DETECTADO



EVIDENCIAS GUARDADAS
Comparativo: kpi_oficiales_II_corte_20261024T190528Z.csv
Auditoría IND-02: IND02_registros_no_normalizados_20261024T190528Z.csv

PASO 4B FINALIZADO
No se modificaron registros de SQL Server.
